# Early Warning of Meteorological Drought in Morocco's Atlas Mountains: A Multi-Paradigm Benchmark of Satellite-Augmented Deep Learning and Tree-Based Models

### Reproduction of the main results (figures and tables) from saved model weights

Nacer Aderdour (1,*), Hassan Rhinane (1), Henri Rueff (2), Mehdi Maanan (1)

1. Department of Geology, Faculty of Science Ain Chock, Hassan II University, Casablanca, Morocco
2. Centre for Development and Environment, University of Bern, Switzerland
(*) Corresponding author: nacer.ader1@gmail.com

This notebook accompanies the manuscript submitted to Theoretical and Applied Climatology.

---

**Overview.** Reproduces the figures and tables reported in the paper. It loads the pre-trained ConvGRU (Model B) weights and runs inference on the held-out test set. No model training is performed, and all outputs are deterministic.

**Requirements.** The `data/`, `weights/`, `configs/`, and `logs/` folders from this repository, and the Python packages in `requirements.txt`.

**Usage.** Run all cells in order. The data-loading step is required because the model consumes the 18-channel feature tensor at inference time.

**Repository.** https://github.com/nacerader/atlas-drought-convgru

## Section 1: Setup & Reproducibility

In [ ]:
# Repository version: input files are read from the local folders set below.
pass

In [ ]:
# -- Configuration ---------------------------------------------------------
# Configuration. Model, training, and split parameters are set in this cell.
# configs/outputs_final_config.json and configs/seed_list.json are
# human-readable documentation snapshots - NOT loaded at runtime.
# configs/best_seed_map.json IS loaded at runtime in Section 6 to
# select the best-validation-loss seed for spatial/case-study outputs.
# If you change a model parameter here, update the JSON for consistency.
# ----------------------------------------------------------------------------
import os, sys, random, warnings, gc, json
from pathlib import Path
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import matplotlib as mpl
from scipy import stats
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tqdm.auto import tqdm
import rasterio
from rasterio.transform import xy

try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    print(f'TensorFlow {tf.__version__}, GPU: {tf.config.list_physical_devices("GPU")}')
except Exception:
    raise RuntimeError('TensorFlow with GPU required')

warnings.filterwarnings('ignore', category=RuntimeWarning)

# Global publication font (set ONCE here)
mpl.rcParams['font.family']      = 'serif'
mpl.rcParams['font.serif']       = ['DejaVu Serif']
mpl.rcParams['mathtext.fontset'] = 'dejavuserif'
mpl.rcParams['axes.titlesize']   = 13
mpl.rcParams['axes.labelsize']   = 11
mpl.rcParams['xtick.labelsize']  = 9
mpl.rcParams['ytick.labelsize']  = 9
mpl.rcParams['legend.fontsize']  = 9
mpl.rcParams['figure.dpi']       = 150

def set_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

# -- PATH OVERRIDES - the only lines you need to edit ------------------------
DATA_DIR    = 'data'
OUTPUT_DIR  = 'outputs'
WEIGHTS_DIR = 'weights'
PACKAGE_DIR = str(Path(WEIGHTS_DIR).parent)
# -----------------------------------------------------------------------------

# Load locked model / training / split params from package JSON files
_pcfg  = json.load(open(Path(PACKAGE_DIR) / 'configs' / 'outputs_final_config.json'))
_seeds = json.load(open(Path(PACKAGE_DIR) / 'configs' / 'seed_list.json'))
def _dt(s): return tuple(int(x) for x in s.split('-'))

cfg = {
    # Environment-specific paths (edit the PATH OVERRIDES block above)
    'DATA_DIR':             DATA_DIR,
    'OUTPUT_DIR':           OUTPUT_DIR,
    'WEIGHTS_DIR':          WEIGHTS_DIR,
    'NODATA':               -9999.0,
    'VARIABLES':            ('precip_mm', 'ndvi', 'lst_c', 't2m_c',
                              'soil_moisture', 'evap_mm', 'elevation', 'slope', 'lst_era5_c'),
    'SEQ_LEN':              6,
    # Data / calibration / splits   from  from outputs_final_config.json
    'SPI_SCALE':            _pcfg['data']['spi_scale'],
    'SPI_CALIB_START':      _dt(_pcfg['data']['spi_calibration']['start']),
    'SPI_CALIB_END':        _dt(_pcfg['data']['spi_calibration']['end']),
    'TRAIN_START':          _dt(_pcfg['splits']['train']['start']),
    'TRAIN_END':            _dt(_pcfg['splits']['train']['end']),
    'VAL_START':            _dt(_pcfg['splits']['val']['start']),
    'VAL_END':              _dt(_pcfg['splits']['val']['end']),
    'TEST_START':           _dt(_pcfg['splits']['test']['start']),
    'TEST_END':             _dt(_pcfg['splits']['test']['end']),
    # Model architecture   from  from outputs_final_config.json
    'GRU_FILTERS':          _pcfg['model']['gru_filters'],
    'GRU_KERNEL':           _pcfg['model']['gru_kernel'],
    'DROPOUT':              _pcfg['model']['dropout'],
    'WEIGHT_DECAY':         _pcfg['model']['weight_decay'],
    # Training   from  from outputs_final_config.json
    'LR':                   _pcfg['training']['lr'],
    'EPOCHS':               _pcfg['training']['epochs'],
    'PATIENCE':             _pcfg['training']['patience'],
    'BATCH_SIZE':           _pcfg['training']['batch_size'],
    # Evaluation   from  from outputs_final_config.json
    'HORIZONS':             tuple(_pcfg['evaluation']['horizons']),
    'DROUGHT_THRESHOLD':    _pcfg['evaluation']['drought_threshold'],
    # Patch training   from  from outputs_final_config.json
    'USE_PATCH_TRAINING':   _pcfg['patches']['use_patch_training'],
    'PATCH_SIZE':           _pcfg['patches']['patch_size'],
    'PATCHES_PER_SAMPLE':   _pcfg['patches']['patches_per_sample'],
    'MIN_PATCH_ROI_FRAC':   _pcfg['patches']['min_roi_frac'],
    'MAX_TARGET_NAN_FRAC':  _pcfg['patches']['max_target_nan_frac'],
    # Seeds   from  from seed_list.json
    'SEEDS':                tuple(_seeds['seeds']),
}
print(f'Config loaded: HORIZONS={cfg["HORIZONS"]}, SEEDS={cfg["SEEDS"]}, '
      f'GRU_FILTERS={cfg["GRU_FILTERS"]}, LR={cfg["LR"]}, EPOCHS={cfg["EPOCHS"]}')

output_dir = Path(cfg['OUTPUT_DIR'])
for d in ['tables', 'figures', 'models', 'data', 'geotiffs']:
    (output_dir / d).mkdir(parents=True, exist_ok=True)

set_seeds(42)
print('Configuration loaded. Output dir:', output_dir)
print(f'NumPy {np.__version__}, Pandas {pd.__version__}, TF {tf.__version__}')

# -- PREDICTION RULE ------------------------------------------------
# Scalar metrics, event metrics, DM tests, time-series figures:
#    ->  ensemble mean across all seeds  (all_test_preds[h])
# Full-grid spatial maps, permutation importance, case study,
# future forecast:
#    ->  best-validation-loss seed  (best_models[h], see configs/best_seed_map.json)
# Both rules are documented in configs/outputs_final_config.json.
# --------------------------------------------------------------------


## Section 2: Data Loading

In [ ]:

def in_range(d, start, end):
    return (start[0], start[1]) <= (d[0], d[1]) <= (end[0], end[1])

def load_yearly_file(filepath, n_vars):
    with rasterio.open(filepath) as src:
        arr = src.read().astype(np.float32)
        prof = {'crs': src.crs, 'transform': src.transform,
                'height': src.height, 'width': src.width}
    n_bands, h, w = arr.shape
    n_months = n_bands // n_vars
    out = np.zeros((n_months, h, w, n_vars), dtype=np.float32)
    for mm in range(n_months):
        for vv in range(n_vars):
            out[mm, :, :, vv] = arr[mm * n_vars + vv]
    return out, prof

all_data_list, all_dates, profile = [], [], None
for year in tqdm(range(1981, 2027), desc='Loading GeoTIFFs'):
    fp = os.path.join(cfg['DATA_DIR'], f'monthly_{year}.tif')
    if not os.path.exists(fp):
        continue
    y_data, p = load_yearly_file(fp, len(cfg['VARIABLES']))
    if profile is None:
        profile = p
    all_data_list.append(y_data)
    for month in range(1, y_data.shape[0] + 1):
        all_dates.append((year, month))

if not all_data_list:
    raise FileNotFoundError(f'No GeoTIFFs in {cfg["DATA_DIR"]}')

all_data = np.concatenate(all_data_list, axis=0)
all_data = np.where(all_data == cfg['NODATA'], np.nan, all_data)
N_MONTHS, HEIGHT, WIDTH, N_VARS = all_data.shape
idx_map = {v: cfg['VARIABLES'].index(v) for v in cfg['VARIABLES']}
ROI_MASK = np.isfinite(all_data[0, :, :, idx_map['elevation']])

print(f'Data: {N_MONTHS} months, {HEIGHT}x{WIDTH}, ROI={ROI_MASK.sum()} pixels ({ROI_MASK.mean()*100:.1f}%)')
print(f'Date range: {all_dates[0]} to {all_dates[-1]}')
print(f'Variables: {list(idx_map.keys())}')

train_mask = np.array([in_range(d, cfg['TRAIN_START'], cfg['TRAIN_END']) for d in all_dates])
val_mask   = np.array([in_range(d, cfg['VAL_START'],   cfg['VAL_END'])   for d in all_dates])
test_mask  = np.array([in_range(d, cfg['TEST_START'],  cfg['TEST_END'])  for d in all_dates])
train_idx  = np.where(train_mask)[0]
val_idx    = np.where(val_mask)[0]
test_idx   = np.where(test_mask)[0]

def rolling_sum_3d(x, window):
    n, h, w = x.shape
    out = np.full_like(x, np.nan)
    for t in range(window - 1, n):
        out[t] = np.nansum(x[t - window + 1:t + 1], axis=0)
    return out

def fit_gamma_per_month(pa, dates, roi, cs, ce, ms=10):
    n, h, w = pa.shape
    params = np.full((12, h, w, 2), np.nan, dtype=np.float32)
    for month in tqdm(range(1, 13), desc='Gamma fit'):
        ci = [i for i, d in enumerate(dates) if d[1] == month and in_range(d, cs, ce)]
        if len(ci) < ms:
            continue
        md = pa[ci]
        ys, xs = np.where(roi)
        for y, x in zip(ys, xs):
            v = md[:, y, x]; v = v[np.isfinite(v)]; v = v[v > 0]
            if len(v) < ms:
                continue
            try:
                a, _, b = stats.gamma.fit(v, floc=0)
                params[month-1, y, x, 0] = a
                params[month-1, y, x, 1] = b
            except Exception:
                pass
    return params

def gamma_to_spi(pa, dates, params, roi):
    n, h, w = pa.shape
    spi = np.full((n, h, w), np.nan, dtype=np.float32)
    for t, (_, month) in enumerate(dates):
        a, b, x = params[month-1,:,:,0], params[month-1,:,:,1], pa[t]
        valid = np.isfinite(x) & np.isfinite(a) & np.isfinite(b) & roi
        pos = valid & (x > 0); non_pos = valid & (x <= 0)
        if np.any(pos):
            cdf = np.clip(stats.gamma.cdf(x[pos], a[pos], scale=b[pos]), 1e-4, 1-1e-4)
            spi[t][pos] = np.clip(stats.norm.ppf(cdf), -4, 4)
        if np.any(non_pos):
            spi[t][non_pos] = -4.0
    return spi

cache = output_dir / 'data' / f'spi{cfg["SPI_SCALE"]}_cache.npz'
alt_caches = [
]
spi6, gamma_params = None, None
for cp in [cache] + alt_caches:
    if cp.exists():
        print(f'Loading cached SPI-{cfg["SPI_SCALE"]} from {cp}')
        _c = np.load(cp)
        spi6, gamma_params = _c['spi'], _c['gamma_params']
        break
if spi6 is None:
    print(f'Computing SPI-{cfg["SPI_SCALE"]} from scratch...')
    precip = all_data[:, :, :, idx_map['precip_mm']]
    precip_acc = rolling_sum_3d(precip, cfg['SPI_SCALE'])
    gamma_params = fit_gamma_per_month(precip_acc, all_dates, ROI_MASK,
                                       cfg['SPI_CALIB_START'], cfg['SPI_CALIB_END'])
    spi6 = gamma_to_spi(precip_acc, all_dates, gamma_params, ROI_MASK)
    np.savez_compressed(cache, spi=spi6, gamma_params=gamma_params)
    print(f'Saved cache: {cache}')

print(f'SPI-{cfg["SPI_SCALE"]} shape: {spi6.shape}')

def monthly_climatology(spi, dates, tr_idx):
    clim = np.full((12, spi.shape[1], spi.shape[2]), np.nan, dtype=np.float32)
    for month in range(1, 13):
        idx = [i for i in tr_idx if dates[i][1] == month]
        if idx:
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                clim[month - 1] = np.nanmean(spi[idx], axis=0)
    return clim

clim = monthly_climatology(spi6, all_dates, train_idx)

def build_anchor_samples(spi, dates, anchor_idx, horizon, seq_len, clim, roi_mask, max_nan=0.4):
    obs, pers, clim_pred, used = [], [], [], []
    for t in anchor_idx:
        if t < seq_len or t + horizon >= len(spi):
            continue
        y = spi[t + horizon]
        if np.isnan(y[roi_mask]).mean() > max_nan:
            continue
        obs.append(y)
        pers.append(spi[t - 1])
        clim_pred.append(clim[dates[t + horizon][1] - 1])
        used.append(t)
    return {'anchor': np.array(used, dtype=np.int32),
            'obs': np.array(obs, dtype=np.float32),
            'persistence': np.array(pers, dtype=np.float32),
            'climatology': np.array(clim_pred, dtype=np.float32)}

def fit_damped_alpha(pack, roi_mask):
    o = pack['obs'][:, roi_mask].ravel()
    p = pack['persistence'][:, roi_mask].ravel()
    c = pack['climatology'][:, roi_mask].ravel()
    m = np.isfinite(o) & np.isfinite(p) & np.isfinite(c)
    x, y = p[m] - c[m], o[m] - c[m]
    return float(np.clip(np.sum(x * y) / (np.sum(x * x) + 1e-12), 0.0, 1.0))

damped_alphas = {}
train_packs = {}
test_packs = {}
for h in cfg['HORIZONS']:
    tr = build_anchor_samples(spi6, all_dates, train_idx, h, cfg['SEQ_LEN'], clim, ROI_MASK)
    te = build_anchor_samples(spi6, all_dates, test_idx,  h, cfg['SEQ_LEN'], clim, ROI_MASK)
    train_packs[h] = tr
    test_packs[h]  = te
    damped_alphas[h] = fit_damped_alpha(tr, ROI_MASK)
    print(f'H={h}: damped alpha={damped_alphas[h]:.4f}, train={len(tr["anchor"])}, test={len(te["anchor"])}')

print('Data loading complete.')


## Section 3: Split & Leakage Check

In [ ]:
print('=== Section 3: Split & Leakage Check ===')

splits = {
    'Train': (train_idx, cfg['TRAIN_START'], cfg['TRAIN_END']),
    'Val':   (val_idx,   cfg['VAL_START'],   cfg['VAL_END']),
    'Test':  (test_idx,  cfg['TEST_START'],  cfg['TEST_END']),
}
for name, (idx, start, end) in splits.items():
    d0 = all_dates[idx[0]]; d1 = all_dates[idx[-1]]
    print(f'{name:6s}: {d0[0]}-{d0[1]:02d} to {d1[0]}-{d1[1]:02d}  ({len(idx)} months)')

assert len(set(train_idx) & set(val_idx))  == 0, 'TRAIN/VAL OVERLAP!'
assert len(set(train_idx) & set(test_idx)) == 0, 'TRAIN/TEST OVERLAP!'
assert len(set(val_idx)   & set(test_idx)) == 0, 'VAL/TEST OVERLAP!'
print('No split overlap confirmed.')
print(f'ROI pixel count: {ROI_MASK.sum()} ({ROI_MASK.mean()*100:.1f}% of grid)')


## Section 4: Feature Engineering

In [ ]:
print('=== Section 4: Feature Engineering ===')

static_full = np.zeros((HEIGHT, WIDTH, 4), dtype=np.float32)
static_full[:, :, 0] = all_data[0, :, :, idx_map['elevation']]
static_full[:, :, 1] = all_data[0, :, :, idx_map['slope']]
transform = profile['transform']
for r in range(HEIGHT):
    for c in range(WIDTH):
        xx, yy = xy(transform, r, c)
        static_full[r, c, 2] = yy
        static_full[r, c, 3] = xx

mu_st = np.array([np.nanmean(static_full[:,:,i][ROI_MASK]) for i in range(4)], dtype=np.float32)
sd_st = np.array([np.nanstd(static_full[:,:,i][ROI_MASK]) for i in range(4)], dtype=np.float32)
sd_st[sd_st == 0] = 1.0

dyn_vars = ['precip_mm', 'soil_moisture', 'lst_era5_c', 'evap_mm']
dyn_idx_list = [idx_map[v] for v in dyn_vars]
dyn_data = all_data[:, :, :, dyn_idx_list].copy()
C = len(dyn_vars)

mu_dyn = np.array([np.nanmean(dyn_data[train_mask][:, ROI_MASK, i]) for i in range(C)], dtype=np.float32)
sd_dyn = np.array([np.nanstd(dyn_data[train_mask][:, ROI_MASK, i]) for i in range(C)], dtype=np.float32)
sd_dyn[sd_dyn == 0] = 1.0

ndvi_data = all_data[:, :, :, idx_map['ndvi']]
ndvi_mu = float(np.nanmean(ndvi_data[train_mask][:, ROI_MASK]))
ndvi_sd = float(np.nanstd(ndvi_data[train_mask][:, ROI_MASK]))
if not np.isfinite(ndvi_sd) or ndvi_sd == 0:
    ndvi_sd = 1.0

mu_spi = float(np.nanmean(spi6[train_mask][:, ROI_MASK]))
sd_spi = float(np.nanstd(spi6[train_mask][:, ROI_MASK]))
if sd_spi == 0: sd_spi = 1.0

# Channels: SPI(1) + dyn(4) + dyn_valid(4) + NDVI(1) + NDVI_valid(1) + season(2) + static(4) + ROI(1) = 18
n_ch = 1 + C + C + 2 + 2 + 4 + 1
feat = np.zeros((N_MONTHS, HEIGHT, WIDTH, n_ch), dtype=np.float32)
k = 0
feat[:,:,:,k] = np.nan_to_num((spi6 - mu_spi) / sd_spi, nan=0.0); k += 1
for i in range(C):
    feat[:,:,:,k] = np.nan_to_num((dyn_data[:,:,:,i] - mu_dyn[i]) / sd_dyn[i], nan=0.0); k += 1
for i in range(C):
    feat[:,:,:,k] = np.isfinite(dyn_data[:,:,:,i]).astype(np.float32); k += 1
feat[:,:,:,k] = np.nan_to_num((ndvi_data - ndvi_mu) / ndvi_sd, nan=0.0); k += 1
feat[:,:,:,k] = np.isfinite(ndvi_data).astype(np.float32); k += 1
for t, (_, m) in enumerate(all_dates):
    feat[t,:,:,k]   = np.sin(2 * np.pi * m / 12)
    feat[t,:,:,k+1] = np.cos(2 * np.pi * m / 12)
k += 2
for i in range(4):
    feat[:,:,:,k] = np.nan_to_num((static_full[:,:,i] - mu_st[i]) / sd_st[i], nan=0.0); k += 1
feat[:,:,:,k] = ROI_MASK.astype(np.float32)

channel_list = (
    ['SPI_norm'] +
    [f'{v}_norm' for v in dyn_vars] +
    [f'{v}_valid' for v in dyn_vars] +
    ['NDVI_norm', 'NDVI_valid', 'sin_month', 'cos_month'] +
    ['elev_norm', 'slope_norm', 'lat_norm', 'lon_norm', 'ROI_mask']
)
print(f'Feature tensor shape: {feat.shape}')
print(f'Channels ({n_ch}): {channel_list}')


## Section 5: Model Architecture

In [ ]:
print('=== Section 5: Model Architecture ===')

@keras.utils.register_keras_serializable(package='Final')
class ConvGRU2D(layers.Layer):
    def __init__(self, filters, kernel_size=3, padding='same',
                 return_sequences=False, dropout=0.0, weight_decay=0.0, **kw):
        super().__init__(**kw)
        self.filters = int(filters)
        self.kernel_size = kernel_size
        self.padding = padding
        self.return_sequences = bool(return_sequences)
        self.dropout = float(dropout)
        self.weight_decay = float(weight_decay)
        reg = keras.regularizers.l2(self.weight_decay) if self.weight_decay > 0 else None
        self.conv_z = layers.Conv2D(self.filters, self.kernel_size, padding=self.padding, kernel_regularizer=reg)
        self.conv_r = layers.Conv2D(self.filters, self.kernel_size, padding=self.padding, kernel_regularizer=reg)
        self.conv_h = layers.Conv2D(self.filters, self.kernel_size, padding=self.padding, kernel_regularizer=reg)
        self.drop = layers.Dropout(self.dropout)
    def get_config(self):
        c = super().get_config()
        c.update({'filters': self.filters, 'kernel_size': self.kernel_size,
                  'padding': self.padding, 'return_sequences': self.return_sequences,
                  'dropout': self.dropout, 'weight_decay': self.weight_decay})
        return c
    def call(self, inputs, training=None):
        batch, h, w = tf.shape(inputs)[0], tf.shape(inputs)[2], tf.shape(inputs)[3]
        h_state = tf.zeros((batch, h, w, self.filters), dtype=inputs.dtype)
        outputs = []
        for x_t in tf.unstack(inputs, axis=1):
            if self.dropout > 0:
                x_t = self.drop(x_t, training=training)
            xh = tf.concat([x_t, h_state], axis=-1)
            z = tf.sigmoid(self.conv_z(xh))
            r = tf.sigmoid(self.conv_r(xh))
            h_tilde = tf.tanh(self.conv_h(tf.concat([x_t, r * h_state], axis=-1)))
            h_state = (1.0 - z) * h_state + z * h_tilde
            outputs.append(h_state)
        return tf.stack(outputs, axis=1) if self.return_sequences else h_state

def masked_mse_factory(roi):
    roi_t = tf.constant(roi[None, :, :, None], dtype=tf.float32)
    def loss(y_true, y_pred):
        valid = tf.cast(tf.math.is_finite(y_true), tf.float32)
        yc = tf.where(tf.math.is_finite(y_true), y_true, 0.0)
        ht, wt = tf.shape(yc)[1], tf.shape(yc)[2]
        rm = tf.cast(tf.squeeze(tf.image.resize(roi_t, [ht, wt], method='nearest'), -1) > 0.5, tf.float32)
        mask = valid * rm
        return tf.reduce_sum(tf.square(yc - y_pred) * mask) / (tf.reduce_sum(mask) + 1e-8)
    return loss

def extract_patches(X, y, ps, n_patches, roi, min_frac, seed=42):
    Xp, yp = [], []
    rng = np.random.default_rng(seed)
    _, T, H, W, Cc = X.shape
    for i in range(len(X)):
        cnt, att = 0, 0
        while cnt < n_patches and att < n_patches * 10:
            r = rng.integers(0, max(1, H - ps)); c = rng.integers(0, max(1, W - ps))
            att += 1
            if roi[r:r+ps, c:c+ps].mean() < min_frac:
                continue
            Xp.append(X[i, :, r:r+ps, c:c+ps, :])
            yp.append(y[i, r:r+ps, c:c+ps])
            cnt += 1
    return np.array(Xp, dtype=np.float32), np.array(yp, dtype=np.float32)

def make_sequences(anchor_idx, horizon, alpha):
    X, yd, ya, base, anc = [], [], [], [], []
    for t in anchor_idx:
        if t < cfg['SEQ_LEN'] or t + horizon >= len(all_dates):
            continue
        y = spi6[t + horizon]
        if np.isnan(y[ROI_MASK]).mean() > cfg['MAX_TARGET_NAN_FRAC']:
            continue
        p = spi6[t - 1]
        c = clim[all_dates[t + horizon][1] - 1]
        b = alpha * p + (1 - alpha) * c
        X.append(feat[t - cfg['SEQ_LEN']:t])
        yd.append(y - b)
        ya.append(y)
        base.append(b)
        anc.append(t)
    return {'X': np.array(X, dtype=np.float32), 'y_delta': np.array(yd, dtype=np.float32),
            'y_abs': np.array(ya, dtype=np.float32), 'baseline': np.array(base, dtype=np.float32),
            'anchor': np.array(anc, dtype=np.int32)}

def build_model():
    inp = layers.Input(shape=(cfg['SEQ_LEN'], None, None, n_ch))
    x = layers.LayerNormalization()(inp)
    x = ConvGRU2D(cfg['GRU_FILTERS'], cfg['GRU_KERNEL'], return_sequences=False,
                  dropout=cfg['DROPOUT'], weight_decay=cfg['WEIGHT_DECAY'], name='gru')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(1, 1, padding='same', activation='linear',
                      kernel_regularizer=keras.regularizers.l2(cfg['WEIGHT_DECAY']))(x)
    out = layers.Lambda(lambda z: tf.squeeze(z, axis=-1))(x)
    return keras.Model(inp, out, name='ConvGRU_ModelB')

_m = build_model()
_m.summary()
del _m
keras.backend.clear_session(); gc.collect()
print('Section 5 complete.')



## Section 6: Load Pre-trained Weights & Run Inference

Loads best-validation-loss weights for each horizon/seed from cfg[WEIGHTS_DIR].  
Runs model.predict() on the test set. Builds ensemble-mean predictions.  
**No training occurs in this notebook.**

In [ ]:
print("=== Section 6: Load Weights & Run Inference ===")

# WEIGHTS_DIR must be defined FIRST before any path operations
WEIGHTS_DIR = Path(cfg["WEIGHTS_DIR"])

# Load authoritative best-seed mapping from config
_bsm_path = WEIGHTS_DIR.parent / 'configs' / 'best_seed_map.json'
if _bsm_path.exists():
    _bsm = json.load(open(_bsm_path))
    _best_seed_from_config = {int(k): int(v) for k, v in _bsm.items() if k.isdigit()}
    print(f"  best_seed_map loaded from config: {_best_seed_from_config}")
else:
    _best_seed_from_config = {}
    print("WARNING: best_seed_map.json not found - will infer from logs")

# Helper metrics (defined here since Section 6 replaces training)
def flatten_metric(obs, pred, roi_mask):
    o, p = obs[:, roi_mask].ravel(), pred[:, roi_mask].ravel()
    m = np.isfinite(o) & np.isfinite(p)
    o, p = o[m], p[m]
    return {"rmse": float(np.sqrt(mean_squared_error(o, p))),
            "mae": float(mean_absolute_error(o, p)),
            "corr": float(np.corrcoef(o, p)[0, 1]) if len(o) > 1 else np.nan,
            "std_ratio": float(np.std(p) / (np.std(o) + 1e-12)),
            "n": int(len(o))}

def drought_scores(obs, pred, roi_mask, thr=-1.0):
    o, p = obs[:, roi_mask].ravel(), pred[:, roi_mask].ravel()
    m = np.isfinite(o) & np.isfinite(p)
    o, p = o[m], p[m]
    tp = np.sum((o < thr) & (p < thr))
    fp = np.sum((o >= thr) & (p < thr))
    fn = np.sum((o < thr) & (p >= thr))
    pod = tp / (tp + fn + 1e-12)
    far = fp / (tp + fp + 1e-12)
    csi = tp / (tp + fp + fn + 1e-12)
    prec = tp / (tp + fp + 1e-12)
    rec = tp / (tp + fn + 1e-12)
    f1 = 2 * prec * rec / (prec + rec + 1e-12)
    return {"pod": float(pod), "far": float(far), "csi": float(csi),
            "precision": float(prec), "recall": float(rec), "f1": float(f1)}

best_models        = {}
best_val_loss_by_h = {}
all_seed_preds     = {}
all_test_preds     = {}

for h in cfg["HORIZONS"]:
    alpha = damped_alphas[h]
    all_seed_preds[h] = []

    # Step 1: determine best seed from logs (fallback only)
    log_best_seed = cfg["SEEDS"][0]
    best_val = np.inf
    for seed in cfg["SEEDS"]:
        for log_dir in [output_dir / "logs", WEIGHTS_DIR.parent / "logs"]:
            log_p = log_dir / ("training_log_h%d_seed%d.csv" % (h, seed))
            if log_p.exists():
                hist = pd.read_csv(log_p)
                mv = float(hist["val_loss"].min())
                if mv < best_val:
                    best_val = mv
                    log_best_seed = seed
                break

    # Step 2: config is authoritative - override log inference if available
    best_seed_for_h = _best_seed_from_config.get(h, log_best_seed)
    print(f"  H={h}: best seed = {best_seed_for_h}"
          f" (source={'config' if h in _best_seed_from_config else 'logs'},"
          f" val_loss={best_val:.6f})")

    for seed in cfg["SEEDS"]:
        wf = WEIGHTS_DIR / ("convgru_h%d_s%d.weights.h5" % (h, seed))
        if not wf.exists():
            print(f"  WARNING: {wf.name} not found - skipping seed {seed}")
            continue

        keras.backend.clear_session(); gc.collect()
        model = build_model()
        model.compile(optimizer=keras.optimizers.Adam(cfg["LR"]),
                      loss=masked_mse_factory(ROI_MASK))
        model.load_weights(str(wf))
        print(f"  H={h} seed={seed}: loaded {wf.name}")

        te         = make_sequences(test_idx, h, alpha)
        pred_delta = model.predict(te["X"], batch_size=1, verbose=0)
        pred_abs   = te["baseline"] + pred_delta

        # Store seed explicitly in each entry so dl_results can use it
        all_seed_preds[h].append({
            "seed":       seed,
            "pred_abs":   pred_abs.copy(),
            "pred_delta": pred_delta.copy(),
            "obs":        te["y_abs"].copy(),
            "baseline":   te["baseline"].copy(),
            "anchor":     te["anchor"].copy(),
        })
        if seed == best_seed_for_h:
            best_models[h] = model
            best_val_loss_by_h[h] = best_val

    all_test_preds[h] = {
        "pred_abs":   np.mean(np.stack([sp["pred_abs"]   for sp in all_seed_preds[h]], axis=0), axis=0),
        "pred_delta": np.mean(np.stack([sp["pred_delta"] for sp in all_seed_preds[h]], axis=0), axis=0),
        "obs":        all_seed_preds[h][0]["obs"],
        "baseline":   all_seed_preds[h][0]["baseline"],
        "anchor":     all_seed_preds[h][0]["anchor"],
    }
    print(f"  H={h}: ensemble from {len(all_seed_preds[h])} seeds")

# Build per-seed metrics dataframe (seed column uses actual seed, not -1)
dl_results = []
for h in cfg["HORIZONS"]:
    for sp in all_seed_preds[h]:
        met = flatten_metric(sp["obs"], sp["pred_abs"], ROI_MASK)
        dr  = drought_scores(sp["obs"], sp["pred_abs"], ROI_MASK, cfg["DROUGHT_THRESHOLD"])
        # Read best_epoch from training log
        best_ep = None
        for log_dir in [output_dir / "logs", WEIGHTS_DIR.parent / "logs"]:
            log_p = log_dir / ("training_log_h%d_seed%d.csv" % (h, sp["seed"]))
            if log_p.exists():
                _lh = pd.read_csv(log_p)
                best_ep = int(_lh["val_loss"].idxmin()) + 1
                break
        dl_results.append({"horizon": h, "seed": sp["seed"], "model": "ConvGRU_B",
                            "best_epoch": best_ep, **met, **dr})
dl_df = pd.DataFrame(dl_results)

print("\nAll weights loaded. Proceeding to figures.")


## Section 7: Learning Curves

In [ ]:
print('=== Section 7: Learning Curves ===')

WONG = ['#2196F3', '#FF9800', '#4CAF50']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Learning Curves - ConvGRU Model B (SPI-6)', fontsize=13, fontweight='bold')

for col, h in enumerate(cfg['HORIZONS']):
    ax = axes[col]
    for i, seed in enumerate(cfg['SEEDS']):
        fname = None
        for _ld in [output_dir / 'tables', WEIGHTS_DIR.parent / 'logs', output_dir / 'logs']:
            for _fn in [f'history_h{h}_s{seed}.csv', f'training_log_h{h}_seed{seed}.csv']:
                _fp = _ld / _fn
                if _fp.exists():
                    fname = _fp; break
            if fname is not None:
                break
        if fname is None:
            print(f'  No history CSV for H={h} seed={seed} - skipping'); continue
        df_h = pd.read_csv(fname)
        epochs = range(1, len(df_h) + 1)
        ax.plot(epochs, df_h['loss'],     color=WONG[i], lw=1.0, ls='-',  alpha=0.5, label=f's{seed} train')
        ax.plot(epochs, df_h['val_loss'], color=WONG[i], lw=1.5, ls='--', alpha=0.9, label=f's{seed} val')
        best_ep = df_h['val_loss'].idxmin() + 1
        ax.axvline(best_ep, color=WONG[i], lw=0.6, ls=':', alpha=0.4)
    ax.set_title(f'H={h}', fontsize=11)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Masked MSE')
    ax.legend(fontsize=7, ncol=2); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / 'figures' / 'fig04_learning_curves.png', dpi=200, bbox_inches='tight')
plt.show()
print('Section 7 complete.')

In [ ]:
# ============================================================
# SECTION 7b: LEARNING CURVES (Enhanced)
# ============================================================
print('=== Section 7: Learning Curves (Statistical Ensembles) ===')

# Color palette
COLOR_TRAIN = '#2E5A88' # Deep Slate Blue
COLOR_VAL = '#C0392B'   # Crimson Red

# Setup Figure
fig, axes = plt.subplots(1, 3, figsize=(18, 7), dpi=300, sharey=True)
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['DejaVu Serif']

for col, h in enumerate(cfg['HORIZONS']):
    ax = axes[col]
    tr_all, va_all = [], []
    best_eps = []

    # 1. Flexible File Loading & Individual Trajectories
    for i, seed in enumerate(cfg['SEEDS']):
        fname = None
        # Searching through possible directories (matches your existing logic)
        for _ld in [output_dir / 'tables', WEIGHTS_DIR.parent / 'logs', output_dir / 'logs']:
            for _fn in [f'history_h{h}_s{seed}.csv', f'training_log_h{h}_seed{seed}.csv']:
                _fp = _ld / _fn
                if _fp.exists():
                    fname = _fp; break
            if fname is not None: break

        if fname is None:
            print(f'  No history CSV for H={h} seed={seed} - skipping'); continue

        df_h = pd.read_csv(fname)
        tr_all.append(df_h['loss'].values)
        va_all.append(df_h['val_loss'].values)
        best_eps.append(df_h['val_loss'].idxmin())

        # Plot individual-seed trajectories
        ax.plot(df_h['loss'].values, color=COLOR_TRAIN, lw=0.8, alpha=0.15, zorder=1)
        ax.plot(df_h['val_loss'].values, color=COLOR_VAL, lw=0.8, alpha=0.15, zorder=1)

    if not tr_all: continue

    # 2. Statistical Averaging (Handles inhomogeneous shapes)
    max_len = max(len(a) for a in tr_all)
    tr_pad = [np.pad(a, (0, max_len - len(a)), constant_values=np.nan) for a in tr_all]
    va_pad = [np.pad(a, (0, max_len - len(a)), constant_values=np.nan) for a in va_all]

    tr_mean = np.nanmean(tr_pad, axis=0)
    va_mean = np.nanmean(va_pad, axis=0)
    epochs = np.arange(max_len)

    # 3. Plot ensemble-mean lines
    ax.plot(epochs, tr_mean, color=COLOR_TRAIN, lw=2.0, ls='--', label='Calibration (Mean)', zorder=3)
    ax.plot(epochs, va_mean, color=COLOR_VAL, lw=3.2, label='Evaluation (Mean)', zorder=4)

    # 4. Selected Model Marker (Average Optimal Point)
    avg_best = int(np.nanmean(best_eps))
    ax.scatter(avg_best, va_mean[avg_best], color='white', edgecolor=COLOR_VAL,
               s=150, lw=3, zorder=10, label='Selected Model' if col==0 else "")
    ax.axvline(avg_best, color='#333333', lw=1.2, ls=':', alpha=0.6, zorder=2)

    # Axis and legend formatting
    ax.set_title(f'Lead Time: H = {h}', fontsize=16, fontweight='bold', pad=15)
    ax.set_xlabel('Epoch', fontweight='bold')
    if col == 0:
        ax.set_ylabel('Masked Mean Squared Error', fontsize=13, fontweight='bold')
        ax.legend(frameon=False, loc='upper right', fontsize=10)

    ax.grid(True, linestyle=':', alpha=0.4, zorder=0)
    for spine in ['top', 'right']: ax.spines[spine].set_visible(False)

plt.tight_layout()
save_path = output_dir / 'figures' / 'fig04_learning_curves_enhanced.png'
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Section 7b complete. Figure saved to: {save_path}')

## Section 8: Baselines

In [ ]:
print('=== Section 8: Baselines ===')

baseline_rows = []

for h in cfg['HORIZONS']:
    te = test_packs[h]
    alpha = damped_alphas[h]
    pred_p = te['persistence']
    pred_c = te['climatology']
    pred_d = alpha * pred_p + (1 - alpha) * pred_c

    for name, pred in [('Persistence', pred_p), ('Climatology', pred_c), ('DampedPers', pred_d)]:
        met = flatten_metric(te['obs'], pred, ROI_MASK)
        dr  = drought_scores(te['obs'], pred, ROI_MASK, cfg['DROUGHT_THRESHOLD'])
        baseline_rows.append({'horizon': h, 'model': name, 'alpha': alpha, **met, **dr})
        print(f'H={h} {name:15s}: RMSE={met["rmse"]:.4f}  Corr={met["corr"]:.4f}  POD={dr["pod"]:.3f}  CSI={dr["csi"]:.3f}')

ys_roi, xs_roi = np.where(ROI_MASK)
n_pixels = len(ys_roi)

def build_ridge_pixel(y_px, x_px, spi, anchor_idx, horizon, seq_len=6):
    feat_r, tgt = [], []
    dyn_idx_r = [idx_map['precip_mm'], idx_map['soil_moisture'], idx_map['lst_era5_c'], idx_map['evap_mm']]
    for t in anchor_idx:
        if t < seq_len or t + horizon >= len(all_dates):
            continue
        tv = spi[t + horizon, y_px, x_px]
        if not np.isfinite(tv):
            continue
        row = []
        for lag in range(1, seq_len + 1):
            v = spi[t - lag, y_px, x_px]; row.append(v if np.isfinite(v) else 0.0)
        for vi in dyn_idx_r:
            v = all_data[t - 1, y_px, x_px, vi]; row.append(v if np.isfinite(v) else 0.0)
        nd = all_data[t - 1, y_px, x_px, idx_map['ndvi']]
        row.append(nd if np.isfinite(nd) else 0.0)
        row.append(1.0 if np.isfinite(nd) else 0.0)
        tm = all_dates[t + horizon][1]
        row.append(np.sin(2 * np.pi * tm / 12))
        row.append(np.cos(2 * np.pi * tm / 12))
        feat_r.append(row); tgt.append(tv)
    if not feat_r:
        return None, None
    return np.array(feat_r, dtype=np.float32), np.array(tgt, dtype=np.float32)

ridge_rows = []
for h in cfg['HORIZONS']:
    print(f'\nRidge H={h}...')
    all_obs_r, all_pred_r = [], []
    for pi in tqdm(range(n_pixels), desc=f'Ridge H={h}', leave=False):
        X_tr_r, y_tr_r = build_ridge_pixel(ys_roi[pi], xs_roi[pi], spi6, train_idx, h)
        X_te_r, y_te_r = build_ridge_pixel(ys_roi[pi], xs_roi[pi], spi6, test_idx, h)
        if X_tr_r is None or X_te_r is None or len(X_tr_r) < 20 or len(X_te_r) < 5:
            continue
        mu_r, sd_r = X_tr_r.mean(0), X_tr_r.std(0); sd_r[sd_r == 0] = 1.0
        mdl = Ridge(alpha=1.0).fit((X_tr_r - mu_r) / sd_r, y_tr_r)
        all_obs_r.extend(y_te_r.tolist())
        all_pred_r.extend(mdl.predict((X_te_r - mu_r) / sd_r).tolist())
    all_obs_r, all_pred_r = np.array(all_obs_r), np.array(all_pred_r)
    rr = float(np.sqrt(mean_squared_error(all_obs_r, all_pred_r)))
    rc = float(np.corrcoef(all_obs_r, all_pred_r)[0, 1]) if len(all_obs_r) > 1 else np.nan
    rm = float(mean_absolute_error(all_obs_r, all_pred_r))
    rs = float(np.std(all_pred_r) / (np.std(all_obs_r) + 1e-12))
    dp_rmse = next(r['rmse'] for r in baseline_rows if r['horizon'] == h and r['model'] == 'DampedPers')
    ridge_rows.append({'horizon': h, 'model': 'Ridge', 'rmse': rr, 'mae': rm,
                       'corr': rc, 'std_ratio': rs, 'vs_dp_pct': (1 - rr / dp_rmse) * 100, 'n': len(all_obs_r)})
    print(f'  RMSE={rr:.4f}  Corr={rc:.4f}  vs DP: {(1-rr/dp_rmse)*100:+.1f}%')

baseline_df = pd.DataFrame(baseline_rows)
ridge_df = pd.DataFrame(ridge_rows)
baseline_df.to_csv(output_dir / 'tables' / 'baselines.csv', index=False)
ridge_df.to_csv(output_dir / 'tables' / 'ridge_results.csv', index=False)
print('Section 8 complete.')


## Section 9: Metrics & Comparison Table

In [ ]:
print('=== Section 9: Metrics & Comparison Table ===')

comp_rows = []
for h in cfg['HORIZONS']:
    for _, r in baseline_df[baseline_df.horizon == h].iterrows():
        comp_rows.append(dict(r))
    for _, r in ridge_df[ridge_df.horizon == h].iterrows():
        rr = dict(r)
        dp_rmse = baseline_df[(baseline_df.horizon == h) & (baseline_df.model == 'DampedPers')].iloc[0]['rmse']
        rr['vs_dp_pct'] = (1 - rr['rmse'] / dp_rmse) * 100
        comp_rows.append(rr)
    subset = dl_df[dl_df.horizon == h]
    if len(subset) > 0:
        dp_rmse = baseline_df[(baseline_df.horizon == h) & (baseline_df.model == 'DampedPers')].iloc[0]['rmse']
        mean_row = {
            'horizon': h, 'model': 'ConvGRU_B (mean)',
            'rmse': subset['rmse'].mean(), 'mae': subset['mae'].mean(),
            'corr': subset['corr'].mean(), 'std_ratio': subset['std_ratio'].mean(),
            'pod': subset['pod'].mean(), 'far': subset['far'].mean(),
            'csi': subset['csi'].mean(), 'f1': subset['f1'].mean(),
            'rmse_std': subset['rmse'].std(), 'corr_std': subset['corr'].std()
        }
        mean_row['vs_dp_pct'] = (1 - mean_row['rmse'] / dp_rmse) * 100
        comp_rows.append(mean_row)

comp_df = pd.DataFrame(comp_rows)
comp_df.to_csv(output_dir / 'tables' / 'comparison_all.csv', index=False)
dl_df.to_csv(output_dir / 'tables' / 'seed_summary.csv', index=False)

for h in cfg['HORIZONS']:
    print(f'\n{"="*70}\nSPI-6  H={h}\n{"="*70}')
    sub = comp_df[comp_df.horizon == h]
    cols = [c for c in ['model', 'rmse', 'mae', 'corr', 'pod', 'far', 'csi', 'vs_dp_pct'] if c in sub.columns]
    print(sub[cols].to_string(index=False, float_format='%.4f'))

print(f'\n{"="*70}\nSKILL SCORE SUMMARY (vs Damped Persistence)\n{"="*70}')
print(f'{"Model":>20} {"H=1":>10} {"H=2":>10} {"H=3":>10}')
for model_name in ['Ridge', 'ConvGRU_B (mean)']:
    scores = []
    for h in cfg['HORIZONS']:
        row = comp_df[(comp_df.horizon == h) & (comp_df.model == model_name)]
        scores.append(f'{row.iloc[0]["vs_dp_pct"]:.1f}%' if len(row) > 0 else 'N/A')
    print(f'{model_name:>20} {scores[0]:>10} {scores[1]:>10} {scores[2]:>10}')

print('Section 9 complete.')

## Section 10: Bootstrap Confidence Intervals

In [ ]:
print('=== Section 10: Bootstrap Confidence Intervals (N=1000) ===')

N_BOOT = 1000
rng_boot = np.random.default_rng(42)
boot_rows = []

for h in cfg['HORIZONS']:
    tp = all_test_preds[h]
    obs = tp['obs']; pred = tp['pred_abs']
    n_t = obs.shape[0]
    rmse_boot, mae_boot, corr_boot, pod_boot, csi_boot = [], [], [], [], []

    for _ in range(N_BOOT):
        idx_b = rng_boot.choice(n_t, n_t, replace=True)
        ob = obs[idx_b][:, ROI_MASK].ravel(); pb = pred[idx_b][:, ROI_MASK].ravel()
        m = np.isfinite(ob) & np.isfinite(pb); ob, pb = ob[m], pb[m]
        if len(ob) < 2: continue
        rmse_boot.append(float(np.sqrt(np.mean((ob - pb)**2))))
        mae_boot.append(float(np.mean(np.abs(ob - pb))))
        corr_boot.append(float(np.corrcoef(ob, pb)[0, 1]))
        tp_b = np.sum((ob < cfg['DROUGHT_THRESHOLD']) & (pb < cfg['DROUGHT_THRESHOLD']))
        fp_b = np.sum((ob >= cfg['DROUGHT_THRESHOLD']) & (pb < cfg['DROUGHT_THRESHOLD']))
        fn_b = np.sum((ob < cfg['DROUGHT_THRESHOLD']) & (pb >= cfg['DROUGHT_THRESHOLD']))
        pod_boot.append(float(tp_b / (tp_b + fn_b + 1e-12)))
        csi_boot.append(float(tp_b / (tp_b + fp_b + fn_b + 1e-12)))

    def ci95(arr):
        return float(np.percentile(arr, 2.5)), float(np.percentile(arr, 97.5))

    r_lo,  r_hi  = ci95(rmse_boot)
    a_lo,  a_hi  = ci95(mae_boot)
    c_lo,  c_hi  = ci95(corr_boot)
    p_lo,  p_hi  = ci95(pod_boot)
    cs_lo, cs_hi = ci95(csi_boot)

    boot_rows.append({'horizon': h,
                      'rmse_mean': np.mean(rmse_boot), 'rmse_ci95_lo': r_lo,  'rmse_ci95_hi': r_hi,
                      'mae_mean':  np.mean(mae_boot),  'mae_ci95_lo':  a_lo,  'mae_ci95_hi':  a_hi,
                      'corr_mean': np.mean(corr_boot), 'corr_ci95_lo': c_lo,  'corr_ci95_hi': c_hi,
                      'pod_mean':  np.mean(pod_boot),  'pod_ci95_lo':  p_lo,  'pod_ci95_hi':  p_hi,
                      'csi_mean':  np.mean(csi_boot),  'csi_ci95_lo':  cs_lo, 'csi_ci95_hi':  cs_hi})
    print(f'H={h}: RMSE={np.mean(rmse_boot):.4f} [{r_lo:.4f},{r_hi:.4f}]  '
          f'Corr={np.mean(corr_boot):.4f} [{c_lo:.4f},{c_hi:.4f}]  '
          f'POD={np.mean(pod_boot):.3f} [{p_lo:.3f},{p_hi:.3f}]  '
          f'CSI={np.mean(csi_boot):.3f} [{cs_lo:.3f},{cs_hi:.3f}]')

boot_df = pd.DataFrame(boot_rows)
boot_df.to_csv(output_dir / 'tables' / 'metrics_bootstrap_ci95.csv', index=False)
print('Saved metrics_bootstrap_ci95.csv')
print('Section 10 complete.')


## Section 11: Diebold-Mariano Test

In [ ]:
print('=== Section 11: Diebold-Mariano Tests ===')

def diebold_mariano(loss_a, loss_b, horizon=1):
    d = np.asarray(loss_a) - np.asarray(loss_b)
    d = d[np.isfinite(d)]
    T = len(d)
    if T < 8:
        return np.nan, np.nan, T
    d_bar = np.mean(d)
    d_ctr = d - d_bar
    lrv = np.sum(d_ctr**2) / T
    for lag in range(1, int(horizon)):
        lrv += 2.0 * np.sum(d_ctr[lag:] * d_ctr[:-lag]) / T
    if not np.isfinite(lrv) or lrv <= 0:
        return np.nan, np.nan, T
    dm = d_bar / np.sqrt(lrv / T)
    corr_f = np.sqrt((T + 1 - 2*horizon + horizon*(horizon-1)/T) / T)
    dm_adj = dm * corr_f
    pval = 2.0 * (1.0 - stats.norm.cdf(np.abs(dm_adj)))
    return float(dm_adj), float(pval), T

def aoi_mse_series(obs, pred, roi):
    s = []
    for i in range(obs.shape[0]):
        o, p = obs[i][roi], pred[i][roi]
        m = np.isfinite(o) & np.isfinite(p)
        s.append(np.mean((o[m] - p[m])**2) if m.sum() > 0 else np.nan)
    return np.array(s)

dm_rows = []
for h in cfg['HORIZONS']:
    tp = all_test_preds[h]
    obs      = tp['obs']
    pred_gru = tp['pred_abs']
    te_pack  = test_packs[h]
    alpha    = damped_alphas[h]
    damped_pred = alpha * te_pack['persistence'] + (1 - alpha) * te_pack['climatology']

    mse_dp   = aoi_mse_series(obs, damped_pred, ROI_MASK)
    mse_gru  = aoi_mse_series(obs, pred_gru, ROI_MASK)
    mse_clim = aoi_mse_series(obs, te_pack['climatology'], ROI_MASK)

    dm_stat, dm_pval, dm_n = diebold_mariano(mse_dp, mse_gru, horizon=h)
    dm_rows.append({'horizon': h, 'comparison': 'ConvGRU_B vs DampedPers',
                    'dm_stat': dm_stat, 'p_value': dm_pval, 'n_timesteps': dm_n,
                    'significant_5pct': dm_pval < 0.05 if np.isfinite(dm_pval) else False})

    dm_stat2, dm_pval2, dm_n2 = diebold_mariano(mse_clim, mse_gru, horizon=h)
    dm_rows.append({'horizon': h, 'comparison': 'ConvGRU_B vs Climatology',
                    'dm_stat': dm_stat2, 'p_value': dm_pval2, 'n_timesteps': dm_n2,
                    'significant_5pct': dm_pval2 < 0.05 if np.isfinite(dm_pval2) else False})

    sig1 = 'YES' if (np.isfinite(dm_pval)  and dm_pval  < 0.05) else 'NO'
    sig2 = 'YES' if (np.isfinite(dm_pval2) and dm_pval2 < 0.05) else 'NO'
    print(f'H={h}: ConvGRU vs DP   DM={dm_stat:+.3f}  p={dm_pval:.4f}  sig={sig1}')
    print(f'H={h}: ConvGRU vs Clim DM={dm_stat2:+.3f}  p={dm_pval2:.4f}  sig={sig2}')

dm_df = pd.DataFrame(dm_rows)
dm_df.to_csv(output_dir / 'tables' / 'diebold_mariano.csv', index=False)
print('\n' + dm_df.to_string(index=False))
print('Section 11 complete.')


## Section 12: Spatial Inference

In [ ]:
# RULE: best-validation-loss seed (best_models[h]) - see configs/best_seed_map.json
print('=== Section 12: Spatial Inference ===')

for h in cfg['HORIZONS']:
    model   = best_models[h]
    tp      = all_test_preds[h]
    anchors = tp['anchor']
    n_test  = len(anchors)
    base_maps = tp['baseline']
    obs_maps  = tp['obs']

    print(f'\nH={h}: generating {n_test} full-grid prediction maps...')
    pred_maps = np.full((n_test, HEIGHT, WIDTH), np.nan, dtype=np.float32)
    for i in tqdm(range(n_test), desc=f'Inference H={h}'):
        t = anchors[i]
        X_full = feat[t - cfg['SEQ_LEN']:t][None, ...]
        pred_delta = model.predict(X_full, batch_size=1, verbose=0)
        pred_maps[i] = base_maps[i] + pred_delta[0]
    pred_maps[:, ~ROI_MASK] = np.nan

    geotiff_path = output_dir / 'geotiffs' / f'spi6_pred_H{h}.tif'
    with rasterio.open(geotiff_path, 'w', driver='GTiff', height=HEIGHT, width=WIDTH,
                       count=n_test, dtype='float32', crs=profile['crs'],
                       transform=profile['transform'], nodata=-9999.0) as dst:
        for i in range(n_test):
            band = pred_maps[i].copy(); band[~np.isfinite(band)] = -9999.0
            dst.write(band, i + 1)
    print(f'  Saved: {geotiff_path}')

    obs_geotiff = output_dir / 'geotiffs' / f'spi6_obs_H{h}.tif'
    with rasterio.open(obs_geotiff, 'w', driver='GTiff', height=HEIGHT, width=WIDTH,
                       count=n_test, dtype='float32', crs=profile['crs'],
                       transform=profile['transform'], nodata=-9999.0) as dst:
        for i in range(n_test):
            band = obs_maps[i].copy(); band[~np.isfinite(band)] = -9999.0
            dst.write(band, i + 1)

    all_test_preds[h]['pred_maps'] = pred_maps
    all_test_preds[h]['obs_maps']  = obs_maps

    spi_cmap = plt.cm.RdYlBu
    spi_norm = mcolors.TwoSlopeNorm(vmin=-3, vcenter=0, vmax=3)
    sample_times = [0, n_test // 2, n_test - 1]

    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    fig.suptitle(f'SPI-6 Forecast Maps - H={h} (top: observed, bottom: predicted)',
                 fontsize=13, fontweight='bold')
    for col, ti in enumerate(sample_times):
        date_str = f'{all_dates[anchors[ti] + h][0]}-{all_dates[anchors[ti] + h][1]:02d}'
        obs_m  = obs_maps[ti].copy();  obs_m[~ROI_MASK]  = np.nan
        pred_m = pred_maps[ti].copy(); pred_m[~ROI_MASK] = np.nan
        im = axes[0, col].imshow(obs_m,  cmap=spi_cmap, norm=spi_norm, aspect='auto')
        axes[0, col].set_title(f'Observed {date_str}', fontsize=10); axes[0, col].axis('off')
        axes[1, col].imshow(pred_m, cmap=spi_cmap, norm=spi_norm, aspect='auto')
        axes[1, col].set_title(f'Predicted {date_str}', fontsize=10); axes[1, col].axis('off')
    plt.colorbar(im, ax=axes, fraction=0.02, pad=0.04, label='SPI-6')
    plt.savefig(output_dir / 'figures' / f'fig07_spatial_maps_H{h}.png', dpi=200, bbox_inches='tight')
    plt.show()

# Per-pixel RMSE map
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Per-Pixel RMSE - ConvGRU Model B (SPI-6)', fontsize=13, fontweight='bold')
for ax, h in zip(axes, cfg['HORIZONS']):
    tp = all_test_preds[h]
    obs_m = tp['obs_maps']; pred_m = tp['pred_maps']
    rm = np.sqrt(np.nanmean((obs_m - pred_m)**2, axis=0)); rm[~ROI_MASK] = np.nan
    im = ax.imshow(rm, cmap='YlOrRd', vmin=0, vmax=2.0, aspect='auto')
    ax.set_title(f'H={h}  (mean={np.nanmean(rm):.3f})', fontsize=11); ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='RMSE')
plt.tight_layout()
plt.savefig(output_dir / 'figures' / 'fig11_spatial_rmse.png', dpi=200, bbox_inches='tight')
plt.show()

# Per-pixel correlation map
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Per-Pixel Correlation - ConvGRU Model B (SPI-6)', fontsize=13, fontweight='bold')
for ax, h in zip(axes, cfg['HORIZONS']):
    tp = all_test_preds[h]
    obs_m = tp['obs_maps']; pred_m = tp['pred_maps']
    cm = np.full((HEIGHT, WIDTH), np.nan, dtype=np.float32)
    for r in range(HEIGHT):
        for c_px in range(WIDTH):
            if not ROI_MASK[r, c_px]: continue
            o = obs_m[:, r, c_px]; p = pred_m[:, r, c_px]
            m = np.isfinite(o) & np.isfinite(p)
            if m.sum() > 3:
                cm[r, c_px] = np.corrcoef(o[m], p[m])[0, 1]
    im = ax.imshow(cm, cmap='RdYlGn', vmin=-0.5, vmax=1.0, aspect='auto')
    ax.set_title(f'H={h}  (median={np.nanmedian(cm):.3f})', fontsize=11); ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='Correlation')
plt.tight_layout()
plt.savefig(output_dir / 'figures' / 'fig12_spatial_correlation.png', dpi=200, bbox_inches='tight')
plt.show()

print('Section 12 complete.')


## Section 13: Drought Occurrence & Categorical Metrics

In [ ]:
print('=== Section 13: Drought Occurrence & Categorical Metrics ===')

thr = cfg['DROUGHT_THRESHOLD']
severity_bins = [(-np.inf, -2.0, 'Extreme'), (-2.0, -1.5, 'Severe'),
                 (-1.5, -1.0, 'Moderate'), (-1.0, np.inf, 'No drought')]
drought_cat_rows = []

for h in cfg['HORIZONS']:
    tp = all_test_preds[h]
    obs_maps  = tp.get('obs_maps', tp['obs'])
    pred_maps = tp.get('pred_maps', tp['pred_abs'])
    anchors   = tp['anchor']
    n_test    = len(anchors)

    obs_drought  = np.where(np.isfinite(obs_maps),  (obs_maps  < thr).astype(np.float32), np.nan)
    pred_drought = np.where(np.isfinite(pred_maps), (pred_maps < thr).astype(np.float32), np.nan)

    pod_ts, far_ts, csi_ts = [], [], []
    for i in range(n_test):
        o = obs_maps[i][ROI_MASK]; p = pred_maps[i][ROI_MASK]
        m = np.isfinite(o) & np.isfinite(p); o, p = o[m], p[m]
        tp_i = np.sum((o < thr) & (p < thr))
        fp_i = np.sum((o >= thr) & (p < thr))
        fn_i = np.sum((o < thr) & (p >= thr))
        pod_ts.append(tp_i / (tp_i + fn_i + 1e-12))
        far_ts.append(fp_i / (tp_i + fp_i + 1e-12))
        csi_ts.append(tp_i / (tp_i + fp_i + fn_i + 1e-12))

    obs_area  = [np.nanmean(obs_drought[i][ROI_MASK])  * 100 for i in range(n_test)]
    pred_area = [np.nanmean(pred_drought[i][ROI_MASK]) * 100 for i in range(n_test)]
    test_dates = [all_dates[a + h] for a in anchors]
    test_dt = pd.to_datetime([f'{d[0]}-{d[1]:02d}' for d in test_dates], format='%Y-%m')

    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    fig.suptitle(f'Drought Occurrence: H={h} (SPI-6 < {thr})', fontsize=13, fontweight='bold')
    axes[0].plot(test_dt, obs_area,  'k-', lw=1.2, label='Observed')
    axes[0].plot(test_dt, pred_area, 'b-', lw=1.0, label='Predicted (ConvGRU B)')
    axes[0].set_ylabel('% ROI area under drought'); axes[0].set_title('Drought Area Fraction')
    axes[0].legend(); axes[0].grid(True, alpha=0.3)
    axes[1].plot(test_dt, pod_ts, 'g-', lw=1.0, label='POD')
    axes[1].plot(test_dt, far_ts, 'r-', lw=1.0, label='FAR')
    axes[1].plot(test_dt, csi_ts, 'b-', lw=1.0, label='CSI')
    axes[1].set_ylabel('Score'); axes[1].set_title('Categorical Metrics Over Time')
    axes[1].set_ylim(-0.05, 1.05); axes[1].legend(); axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(output_dir / 'figures' / f'fig08_drought_occurrence_H{h}.png', dpi=200, bbox_inches='tight')
    plt.show()

    o_all = obs_maps[:, ROI_MASK].ravel(); p_all = pred_maps[:, ROI_MASK].ravel()
    m = np.isfinite(o_all) & np.isfinite(p_all); o_all, p_all = o_all[m], p_all[m]
    for sev_lo, sev_hi, sev_name in severity_bins:
        obs_in  = (o_all >= sev_lo) & (o_all < sev_hi)
        pred_in = (p_all >= sev_lo) & (p_all < sev_hi)
        tp_s = np.sum(obs_in & pred_in); fp_s = np.sum(~obs_in & pred_in); fn_s = np.sum(obs_in & ~pred_in)
        drought_cat_rows.append({'horizon': h, 'severity': sev_name,
                                 'threshold': f'[{sev_lo},{sev_hi})',
                                 'n_obs': int(obs_in.sum()), 'n_pred': int(pred_in.sum()),
                                 'pod': float(tp_s/(tp_s+fn_s+1e-12)),
                                 'far': float(fp_s/(tp_s+fp_s+1e-12)),
                                 'csi': float(tp_s/(tp_s+fp_s+fn_s+1e-12))})

    overall = drought_scores(obs_maps, pred_maps, ROI_MASK, thr)
    print(f'H={h}: POD={overall["pod"]:.3f}  FAR={overall["far"]:.3f}  CSI={overall["csi"]:.3f}')

drought_cat_df = pd.DataFrame(drought_cat_rows)
drought_cat_df.to_csv(output_dir / 'tables' / 'drought_categorical_metrics.csv', index=False)
print(drought_cat_df.to_string(index=False, float_format='%.3f'))
print('Section 13 complete.')


## Section 14: Prediction vs Observed Time Series

In [ ]:
print('=== Section 14: Prediction vs Observed Time Series ===')

ys_sample, xs_sample = np.where(ROI_MASK)
n_roi = len(ys_sample)
sample_indices = [0, n_roi // 4, n_roi // 2, 3 * n_roi // 4]
sample_pixels  = [(ys_sample[i], xs_sample[i]) for i in sample_indices]

for h in cfg['HORIZONS']:
    tp = all_test_preds[h]
    obs = tp['obs']; pred = tp['pred_abs']; base = tp['baseline']
    anchors = tp['anchor']
    test_dates = [all_dates[a + h] for a in anchors]
    test_dt = pd.to_datetime([f'{d[0]}-{d[1]:02d}' for d in test_dates], format='%Y-%m')

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle(f'Predicted vs Observed SPI-6: H={h} (sample pixels)', fontsize=13, fontweight='bold')
    for ax, (yp, xp) in zip(axes.ravel(), sample_pixels):
        obs_px  = obs[:, yp, xp]; pred_px = pred[:, yp, xp]; base_px = base[:, yp, xp]
        valid = np.isfinite(obs_px) & np.isfinite(pred_px)
        ax.plot(test_dt[valid], obs_px[valid],  'k-',  lw=1.2, label='Observed',     alpha=0.8)
        ax.plot(test_dt[valid], pred_px[valid], 'b-',  lw=1.0, label='ConvGRU B',    alpha=0.8)
        ax.plot(test_dt[valid], base_px[valid], 'r--', lw=0.8, label='Damped Pers.', alpha=0.6)
        ax.axhline(cfg['DROUGHT_THRESHOLD'], color='red', lw=0.6, ls=':', alpha=0.5)
        ax.fill_between(test_dt[valid], cfg['DROUGHT_THRESHOLD'], obs_px[valid],
                        where=obs_px[valid] < cfg['DROUGHT_THRESHOLD'],
                        alpha=0.15, color='red', label='Drought')
        rmse_px = np.sqrt(np.nanmean((obs_px[valid] - pred_px[valid])**2))
        corr_px = np.corrcoef(obs_px[valid], pred_px[valid])[0, 1] if valid.sum() > 2 else np.nan
        ax.set_title(f'Pixel ({yp},{xp}) - RMSE={rmse_px:.3f}, r={corr_px:.3f}', fontsize=10)
        ax.set_ylabel('SPI-6'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(output_dir / 'figures' / f'fig05_pred_vs_obs_H{h}.png', dpi=200, bbox_inches='tight')
    plt.show()

# Scatter plots
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Observed vs Predicted SPI-6 (all ROI pixels)', fontsize=13, fontweight='bold')
for ax, h in zip(axes, cfg['HORIZONS']):
    tp = all_test_preds[h]
    o = tp['obs'][:, ROI_MASK].ravel(); p = tp['pred_abs'][:, ROI_MASK].ravel()
    m = np.isfinite(o) & np.isfinite(p); o, p = o[m], p[m]
    if len(o) > 50000:
        idx_s = np.random.default_rng(42).choice(len(o), 50000, replace=False)
        o_pl, p_pl = o[idx_s], p[idx_s]
    else:
        o_pl, p_pl = o, p
    ax.hexbin(o_pl, p_pl, gridsize=60, cmap='Blues', mincnt=1)
    ax.plot([-3, 3], [-3, 3], 'r--', lw=1, label='1:1 line')
    rmse_a = np.sqrt(np.mean((o - p)**2)); corr_a = np.corrcoef(o, p)[0, 1]
    ax.set_title(f'H={h} - RMSE={rmse_a:.3f}, r={corr_a:.3f}', fontsize=11)
    ax.set_xlabel('Observed SPI-6'); ax.set_ylabel('Predicted SPI-6')
    ax.legend(); ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5); ax.set_aspect('equal')
plt.tight_layout()
plt.savefig(output_dir / 'figures' / 'fig06_scatter_obs_vs_pred.png', dpi=200, bbox_inches='tight')
plt.show()

# ROI-mean time series
dates_all_pd = pd.to_datetime([f'{d[0]}-{d[1]:02d}' for d in all_dates], format='%Y-%m')
roi_spi_hist = np.array([np.nanmean(spi6[t][ROI_MASK]) for t in range(N_MONTHS)])

fig, axes = plt.subplots(3, 1, figsize=(16, 14))
fig.suptitle('Historical SPI-6 + ConvGRU Test Predictions', fontsize=14, fontweight='bold')
for ax, h in zip(axes, cfg['HORIZONS']):
    tp = all_test_preds[h]
    pred_m = tp.get('pred_maps', tp['pred_abs']); anchors = tp['anchor']
    test_dates_h = [all_dates[a + h] for a in anchors]
    test_dt_h = pd.to_datetime([f'{d[0]}-{d[1]:02d}' for d in test_dates_h], format='%Y-%m')
    pred_roi = [np.nanmean(pred_m[i][ROI_MASK]) for i in range(len(anchors))]
    ax.plot(dates_all_pd, roi_spi_hist, color='black', lw=0.8, alpha=0.7, label='Observed SPI-6')
    ax.plot(test_dt_h, pred_roi, color='#e65100', lw=1.8, alpha=0.9, label=f'ConvGRU H={h} (test)')
    ax.axhline(cfg['DROUGHT_THRESHOLD'], color='red', lw=0.6, ls=':', alpha=0.4)
    ax.fill_between(dates_all_pd, cfg['DROUGHT_THRESHOLD'], roi_spi_hist,
                    where=roi_spi_hist < cfg['DROUGHT_THRESHOLD'], alpha=0.1, color='red')
    ax.set_ylabel('SPI-6 (ROI mean)'); ax.set_title(f'H={h}', fontsize=11)
    ax.legend(loc='lower left', fontsize=9); ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig(output_dir / 'figures' / 'fig_roi_timeseries.png', dpi=200, bbox_inches='tight')
plt.show()

print('Section 14 complete.')


In [ ]:
# ============================================================
# FONT SETUP - Run once per Colab session before Section 14
# ============================================================
import subprocess
subprocess.run(['apt-get', 'install', '-y', 'ttf-mscorefonts-installer'],
               capture_output=True)
subprocess.run(['fc-cache', '-fv'], capture_output=True)

import matplotlib.font_manager as fm
fm._load_fontmanager(try_read_cache=False)

available = [f.name for f in fm.fontManager.ttflist]
if 'Times New Roman' in available:
    SERIF_FONT = 'Times New Roman'
    print('Times New Roman: FOUND')
elif 'DejaVu Serif' in available:
    SERIF_FONT = 'DejaVu Serif'
    print('Times New Roman not found, using DejaVu Serif')
else:
    SERIF_FONT = 'serif'
    print('Using generic serif fallback')

In [ ]:
print('=== Section 14: Prediction vs Observed Time Series ===')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams['font.family']      = SERIF_FONT
plt.rcParams['axes.titlesize']   = 18
plt.rcParams['axes.labelsize']   = 16
plt.rcParams['xtick.labelsize']  = 14
plt.rcParams['ytick.labelsize']  = 14
plt.rcParams['legend.fontsize']  = 13
plt.rcParams['figure.titlesize'] = 20

ys_sample, xs_sample = np.where(ROI_MASK)
n_roi = len(ys_sample)
sample_indices = [0, n_roi // 4, n_roi // 2, 3 * n_roi // 4]
sample_pixels  = [(ys_sample[i], xs_sample[i]) for i in sample_indices]

for h in cfg['HORIZONS']:
    tp = all_test_preds[h]
    obs  = tp['obs']
    pred = tp['pred_abs']
    base = tp['baseline']
    anchors    = tp['anchor']
    test_dates = [all_dates[a + h] for a in anchors]
    test_dt    = pd.to_datetime(
        [f'{d[0]}-{d[1]:02d}' for d in test_dates], format='%Y-%m'
    )

    fig, axes = plt.subplots(2, 2, figsize=(16, 11))
    fig.suptitle(
        f'Predicted vs Observed SPI-6: H={h} (sample pixels)',
        fontsize=20, fontweight='bold', y=1.01
    )

    for ax, (yp, xp) in zip(axes.ravel(), sample_pixels):
        obs_px  = obs[:, yp, xp]
        pred_px = pred[:, yp, xp]
        base_px = base[:, yp, xp]
        valid   = np.isfinite(obs_px) & np.isfinite(pred_px)

        ax.plot(test_dt[valid], obs_px[valid],  'k-',  lw=2.0,
                label='Observed',     alpha=0.9)
        ax.plot(test_dt[valid], pred_px[valid], 'b-',  lw=1.6,
                label='ConvGRU',      alpha=0.85)
        ax.plot(test_dt[valid], base_px[valid], 'r--', lw=1.3,
                label='Damped Pers.', alpha=0.7)

        ax.axhline(cfg['DROUGHT_THRESHOLD'], color='red',
                   lw=0.8, ls=':', alpha=0.5)
        ax.fill_between(
            test_dt[valid], cfg['DROUGHT_THRESHOLD'], obs_px[valid],
            where=obs_px[valid] < cfg['DROUGHT_THRESHOLD'],
            alpha=0.15, color='red', label='Drought'
        )

        rmse_px = np.sqrt(np.nanmean((obs_px[valid] - pred_px[valid])**2))
        corr_px = (
            np.corrcoef(obs_px[valid], pred_px[valid])[0, 1]
            if valid.sum() > 2 else np.nan
        )

        ax.set_title(
            f'Pixel ({yp},{xp}): RMSE= {rmse_px:.3f}, r= {corr_px:.3f}',
            fontsize=16, pad=8
        )
        ax.set_ylabel('SPI-6', fontsize=15)
        ax.tick_params(axis='both', labelsize=13)
        ax.legend(fontsize=12, loc='upper left', framealpha=0.7)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(
        output_dir / 'figures' / f'fig05_pred_vs_obs_H{h}.png',
        dpi=500, bbox_inches='tight'
    )
    plt.show()
    print(f'  H={h} saved.')

print('Section 14 complete.')

In [ ]:
# ROI-mean time series
dates_all_pd = pd.to_datetime(
    [f'{d[0]}-{d[1]:02d}' for d in all_dates], format='%Y-%m'
)
roi_spi_hist = np.array([np.nanmean(spi6[t][ROI_MASK]) for t in range(N_MONTHS)])

plt.rcParams['font.family']      = SERIF_FONT
plt.rcParams['axes.titlesize']   = 16
plt.rcParams['axes.labelsize']   = 15
plt.rcParams['xtick.labelsize']  = 13
plt.rcParams['ytick.labelsize']  = 13
plt.rcParams['legend.fontsize']  = 12
plt.rcParams['figure.titlesize'] = 18

fig, axes = plt.subplots(3, 1, figsize=(18, 14))
fig.suptitle(
    'Historical SPI-6 + ConvGRU Test Predictions',
    fontsize=18, fontweight='bold'
)

for ax, h in zip(axes, cfg['HORIZONS']):
    tp = all_test_preds[h]
    pred_m   = tp.get('pred_maps', tp['pred_abs'])
    anchors  = tp['anchor']
    test_dates_h = [all_dates[a + h] for a in anchors]
    test_dt_h    = pd.to_datetime(
        [f'{d[0]}-{d[1]:02d}' for d in test_dates_h], format='%Y-%m'
    )
    pred_roi = [np.nanmean(pred_m[i][ROI_MASK]) for i in range(len(anchors))]

    ax.plot(dates_all_pd, roi_spi_hist,
            color='black', lw=1.2, alpha=0.75, label='Observed SPI-6')
    ax.plot(test_dt_h, pred_roi,
            color='#e65100', lw=2.2, alpha=0.9, label=f'ConvGRU H={h} (test)')
    ax.axhline(cfg['DROUGHT_THRESHOLD'],
               color='red', lw=0.8, ls=':', alpha=0.4)
    ax.fill_between(
        dates_all_pd, cfg['DROUGHT_THRESHOLD'], roi_spi_hist,
        where=roi_spi_hist < cfg['DROUGHT_THRESHOLD'],
        alpha=0.1, color='red'
    )
    ax.set_ylabel('SPI-6 (ROI mean)', fontsize=15)
    ax.set_title(f'H={h}', fontsize=16)
    ax.tick_params(axis='both', labelsize=13)
    ax.legend(loc='lower left', fontsize=12, framealpha=0.7)
    ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig(
    output_dir / 'figures' / 'fig_roi_timeseries.png',
    dpi=500, bbox_inches='tight'
)
plt.show()

## Section 15: Permutation Feature Importance

In [ ]:
print('=== Section 15: Permutation Feature Importance ===')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm

plt.rcParams['font.family']      = SERIF_FONT
plt.rcParams['axes.titlesize']   = 24
plt.rcParams['axes.labelsize']   = 22
plt.rcParams['xtick.labelsize']  = 20
plt.rcParams['ytick.labelsize']  = 20
plt.rcParams['legend.fontsize']  = 18

# Define channel_groups for permutation importance
# Indices are based on the 'channel_list' from Section 4
channel_groups = {
    'SPI': [0], # 'SPI_norm'
    'Precipitation': [1, 5], # 'precip_mm_norm', 'precip_mm_valid'
    'Soil_Moisture': [2, 6], # 'soil_moisture_norm', 'soil_moisture_valid'
    'LST_ERA5': [3, 7],      # 'lst_era5_c_norm', 'lst_era5_c_valid'
    'Evapotranspiration': [4, 8], # 'evap_mm_norm', 'evap_mm_valid'
    'NDVI': [9, 10],         # 'NDVI_norm', 'NDVI_valid'
    'Seasonality': [11, 12], # 'sin_month', 'cos_month'
    'Static_Geographic': [13, 14, 15, 16], # 'elev_norm', 'slope_norm', 'lat_norm', 'lon_norm'
    'ROI_Mask': [17] # 'ROI_mask'
}

h_imp     = 1
model_imp = best_models[h_imp]
tp        = all_test_preds[h_imp]
obs_imp   = tp.get('obs_maps', tp['obs'])

n_perm_samples = min(20, len(tp['anchor']))
rng_p          = np.random.default_rng(42)
perm_idx       = rng_p.choice(len(tp['anchor']), n_perm_samples, replace=False)

X_perm_list = []
for i in perm_idx:
    t = tp['anchor'][i]
    X_perm_list.append(feat[t - cfg['SEQ_LEN']:t])
X_perm  = np.array(X_perm_list, dtype=np.float32)
base_pm = tp['baseline'][perm_idx]
obs_pm  = obs_imp[perm_idx]

pred_sub     = model_imp.predict(X_perm, batch_size=1, verbose=0)
pred_abs_sub = base_pm + pred_sub
sub_mse      = np.nanmean((obs_pm[:, ROI_MASK] - pred_abs_sub[:, ROI_MASK])**2)

N_REPEATS       = 5
importance_rows = []

for gname, gidx in tqdm(channel_groups.items(), desc='Permutation'):
    mse_increases = []
    for rep in range(N_REPEATS):
        X_shuffled = X_perm.copy()
        for ch in gidx:
            perm_order = rng_p.permutation(n_perm_samples)
            X_shuffled[:, :, :, :, ch] = X_perm[perm_order, :, :, :, ch]
        pred_shuf     = model_imp.predict(X_shuffled, batch_size=1, verbose=0)
        pred_abs_shuf = base_pm + pred_shuf
        shuf_mse      = np.nanmean((obs_pm[:, ROI_MASK] - pred_abs_shuf[:, ROI_MASK])**2)
        mse_increases.append(shuf_mse - sub_mse)
    mean_inc = np.mean(mse_increases)
    std_inc  = np.std(mse_increases)
    importance_rows.append({
        'feature_group':       gname,
        'channels':            str(gidx),
        'mse_increase':        mean_inc,
        'mse_increase_std':    std_inc,
        'relative_importance': mean_inc / (sub_mse + 1e-12) * 100
    })
    print(f'  {gname:35s}: {mean_inc:+.6f} ({mean_inc/(sub_mse+1e-12)*100:+.1f}%)')

imp_df = pd.DataFrame(importance_rows).sort_values('mse_increase', ascending=False)
imp_df.to_csv(output_dir / 'tables' / 'permutation_importance.csv', index=False)

fig, ax = plt.subplots(figsize=(20, 11))
colors = ['#d73027' if v > 0 else '#4575b4' for v in imp_df['relative_importance']]
ax.barh(
    imp_df['feature_group'],
    imp_df['relative_importance'],
    color=colors,
    xerr=imp_df['mse_increase_std'] / (sub_mse + 1e-12) * 100,
    capsize=6,
    height=0.65
)
ax.set_xlabel('Relative MSE Increase (%)', fontsize=22, fontweight='bold')
ax.set_title(
    f'Permutation Feature Importance - ConvGRU Model B (H={h_imp})',
    fontsize=24, fontweight='bold', pad=16
)
ax.tick_params(axis='both', labelsize=20)
ax.axvline(0, color='black', lw=1.0)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(
    output_dir / 'figures' / 'fig10_permutation_importance.png',
    dpi=500, bbox_inches='tight'
)
plt.show()
print('Section 15 complete.')

In [ ]:
# ============================================================
# CELL 15B: Cross-horizon feature-importance trend
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.patheffects as path_effects
import numpy as np
import pandas as pd
from tqdm import tqdm

print('=== Cross-horizon feature-importance trend ===')

plt.rcParams['font.family']      = SERIF_FONT
plt.rcParams['axes.labelsize']   = 22
plt.rcParams['xtick.labelsize']  = 20
plt.rcParams['ytick.labelsize']  = 20
plt.rcParams['legend.fontsize']  = 18

horizons      = [1, 2, 3]
target_groups = ['Precipitation', 'NDVI']
results_trend = {g: [] for g in target_groups}

n_samples = min(30, len(all_test_preds[1]['anchor']))
rng       = np.random.default_rng(42)

for h in horizons:
    print(f"  Computing importance for H={h}...")
    model_h  = best_models[h]
    tp       = all_test_preds[h]
    idx      = rng.choice(len(tp['anchor']), n_samples, replace=False)
    X_sub    = np.array([feat[t - cfg['SEQ_LEN']:t] for t in tp['anchor'][idx]])
    base_sub = tp['baseline'][idx]
    obs_sub  = tp['obs'][idx]
    p_base   = model_h.predict(X_sub, batch_size=1, verbose=0)
    mse_base = np.nanmean(
        (obs_sub[:, ROI_MASK] - (base_sub + p_base)[:, ROI_MASK])**2
    )
    for gname in target_groups:
        gidx   = channel_groups[gname]
        X_shuf = X_sub.copy()
        for ch in gidx:
            X_shuf[:, :, :, :, ch] = X_sub[rng.permutation(n_samples), :, :, :, ch]
        p_shuf   = model_h.predict(X_shuf, batch_size=1, verbose=0)
        mse_shuf = np.nanmean(
            (obs_sub[:, ROI_MASK] - (base_sub + p_shuf)[:, ROI_MASK])**2
        )
        rel_inc = ((mse_shuf - mse_base) / (mse_base + 1e-12)) * 100
        results_trend[gname].append(rel_inc)

# --- DUAL-AXIS PLOT ---
fig, ax1 = plt.subplots(figsize=(20, 11), dpi=300)

color_p = '#d73027'
ax1.set_xlabel('Forecast Horizon (Months)', fontweight='bold', fontsize=22)
ax1.set_ylabel('Precipitation Importance (%)', color=color_p,
               fontweight='bold', fontsize=22)
ln1 = ax1.plot(
    horizons, results_trend['Precipitation'],
    color=color_p, marker='o', lw=4.0, ms=16,
    label='Precipitation (Primary Driver)',
    path_effects=[path_effects.withStroke(linewidth=7, foreground='white')]
)
ax1.tick_params(axis='y', labelcolor=color_p, labelsize=20)
ax1.tick_params(axis='x', labelsize=20)

ax2     = ax1.twinx()
color_n = '#4575b4'
ax2.set_ylabel('NDVI Importance (%)', color=color_n,
               fontweight='bold', fontsize=22)
ln2 = ax2.plot(
    horizons, results_trend['NDVI'],
    color=color_n, marker='D', lw=4.0, ms=16, ls='--',
    label='NDVI (Vegetation Memory)',
    path_effects=[path_effects.withStroke(linewidth=7, foreground='white')]
)
ax2.tick_params(axis='y', labelcolor=color_n, labelsize=20)

# Shaded zone within axis limits
ax1.axvspan(2.5, 3.0, color='gray', alpha=0.12)

# Annotation - safe position, horizontal, inside plot
y_lo, y_hi = ax1.get_ylim()
ax1.text(
    2.62, y_lo + (y_hi - y_lo) * 0.06,
    'Predictability\nShift',
    ha='center', va='bottom',
    fontsize=16, color='gray', fontweight='bold',
    rotation=0
)

plt.title(
    'Decaying Climate Signal vs. Rising Vegetation Response',
    fontsize=24, fontweight='bold', pad=22
)
ax1.set_xticks(horizons)
ax1.set_xticklabels(['H=1 (Short)', 'H=2 (Med)', 'H=3 (Long)'], fontsize=20)
ax1.set_xlim(0.7, 3.3)
ax1.grid(True, linestyle=':', alpha=0.5)

lines  = ln1 + ln2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper left', fontsize=18, frameon=True)

fig.tight_layout()
plt.savefig(
    output_dir / 'figures' / 'fig10_ndvi_defense_trend.png',
    dpi=500, bbox_inches='tight'
)
plt.show()
print('Cross-horizon feature-importance analysis complete.')

## Section 16: Case Study

In [ ]:
# NOTE: The case study event (May 2019) is from the VALIDATION period,
# not the held-out test set. It serves as an illustrative example only
# and should not be interpreted as held-out test evidence.
# RULE: best-validation-loss seed (best_models[h]) - see configs/best_seed_map.json
print('=== Section 16: Drought Case Study ===')

val_drought_scores = []
for t in val_idx:
    if t < cfg['SEQ_LEN']: continue
    roi_mean_spi = np.nanmean(spi6[t][ROI_MASK])
    drought_frac = np.nanmean(spi6[t][ROI_MASK] < cfg['DROUGHT_THRESHOLD']) * 100
    if np.isfinite(roi_mean_spi):
        val_drought_scores.append({'time_idx': t, 'year': all_dates[t][0], 'month': all_dates[t][1],
                                   'roi_mean_spi': roi_mean_spi, 'drought_area_pct': drought_frac})

val_drought_df = pd.DataFrame(val_drought_scores).sort_values('roi_mean_spi')
print('Most severe drought months in validation period:')
print(val_drought_df.head(10).to_string(index=False, float_format='%.3f'))

worst = val_drought_df.iloc[0]
drought_t   = int(worst['time_idx'])
drought_date = all_dates[drought_t]
drought_str  = f'{drought_date[0]}-{drought_date[1]:02d}'
print(f'\nSelected drought event: {drought_str}  ROI mean SPI-6={worst["roi_mean_spi"]:.3f}  Area={worst["drought_area_pct"]:.1f}%')

case_results = {}
spi_cmap = plt.cm.RdYlBu
spi_norm = mcolors.TwoSlopeNorm(vmin=-3, vcenter=0, vmax=3)

for h in cfg['HORIZONS']:
    anchor_t = drought_t - h
    if anchor_t < cfg['SEQ_LEN']: continue
    model = best_models[h]; alpha = damped_alphas[h]
    X_case = feat[anchor_t - cfg['SEQ_LEN']:anchor_t][None, ...]
    persistence  = spi6[anchor_t - 1]
    clim_m       = clim[all_dates[drought_t][1] - 1]
    baseline_case = alpha * persistence + (1 - alpha) * clim_m
    pred_delta = model.predict(X_case, batch_size=1, verbose=0)
    pred_abs   = baseline_case + pred_delta[0]; pred_abs[~ROI_MASK] = np.nan
    obs_map = spi6[drought_t].copy(); obs_map[~ROI_MASK] = np.nan
    dp_map  = baseline_case.copy();  dp_map[~ROI_MASK]  = np.nan
    o = obs_map[ROI_MASK]; p = pred_abs[ROI_MASK]; d = dp_map[ROI_MASK]
    m = np.isfinite(o) & np.isfinite(p)
    rmse_gru = np.sqrt(np.mean((o[m] - p[m])**2)); rmse_dp = np.sqrt(np.mean((o[m] - d[m])**2))
    corr_gru = np.corrcoef(o[m], p[m])[0, 1]
    obs_d = o[m] < cfg['DROUGHT_THRESHOLD']; pred_d = p[m] < cfg['DROUGHT_THRESHOLD']; dp_d = d[m] < cfg['DROUGHT_THRESHOLD']
    tp_g = np.sum(obs_d & pred_d); fp_g = np.sum(~obs_d & pred_d); fn_g = np.sum(obs_d & ~pred_d)
    tp_dp = np.sum(obs_d & dp_d);  fp_dp = np.sum(~obs_d & dp_d);  fn_dp = np.sum(obs_d & ~dp_d)
    case_results[h] = {'obs': obs_map, 'pred': pred_abs, 'dp': dp_map,
                       'rmse_gru': rmse_gru, 'rmse_dp': rmse_dp, 'corr_gru': corr_gru,
                       'pod_gru': tp_g/(tp_g+fn_g+1e-12),  'csi_gru': tp_g/(tp_g+fp_g+fn_g+1e-12),
                       'pod_dp':  tp_dp/(tp_dp+fn_dp+1e-12),'csi_dp':  tp_dp/(tp_dp+fp_dp+fn_dp+1e-12)}
    anchor_date = all_dates[anchor_t]
    print(f'H={h}: issued {anchor_date[0]}-{anchor_date[1]:02d} -> {drought_str}  ConvGRU RMSE={rmse_gru:.4f}  POD={case_results[h]["pod_gru"]:.3f}  DP RMSE={rmse_dp:.4f}  POD={case_results[h]["pod_dp"]:.3f}')

n_h = len(case_results)
fig, axes = plt.subplots(3, n_h, figsize=(6 * n_h, 14))
if n_h == 1: axes = axes[:, None]
fig.suptitle(f'Case Study: Drought Event {drought_str}  ROI mean SPI-6={worst["roi_mean_spi"]:.2f}  {worst["drought_area_pct"]:.0f}% area under drought',
             fontsize=13, fontweight='bold')
for col, h in enumerate(sorted(case_results.keys())):
    cr = case_results[h]
    im = axes[0, col].imshow(cr['obs'],  cmap=spi_cmap, norm=spi_norm, aspect='auto')
    axes[0, col].set_title(f'Observed {drought_str}', fontsize=11); axes[0, col].axis('off')
    axes[1, col].imshow(cr['dp'],   cmap=spi_cmap, norm=spi_norm, aspect='auto')
    axes[1, col].set_title(f'Damped Pers. (H={h})\nRMSE={cr["rmse_dp"]:.3f}  POD={cr["pod_dp"]:.2f}', fontsize=10); axes[1, col].axis('off')
    axes[2, col].imshow(cr['pred'], cmap=spi_cmap, norm=spi_norm, aspect='auto')
    axes[2, col].set_title(f'ConvGRU B (H={h})\nRMSE={cr["rmse_gru"]:.3f}  r={cr["corr_gru"]:.2f}  POD={cr["pod_gru"]:.2f}', fontsize=10); axes[2, col].axis('off')
plt.colorbar(im, ax=axes, fraction=0.015, pad=0.04, label='SPI-6')
plt.tight_layout()
plt.savefig(output_dir / 'figures' / f'fig17_case_study_drought_{drought_str}.png', dpi=200, bbox_inches='tight')
plt.show()

case_table = [{'horizon': h, 'target': drought_str,
               'ConvGRU_RMSE': cr['rmse_gru'], 'DP_RMSE': cr['rmse_dp'],
               'RMSE_improvement_%': (1 - cr['rmse_gru'] / cr['rmse_dp']) * 100,
               'ConvGRU_corr': cr['corr_gru'],
               'ConvGRU_POD': cr['pod_gru'], 'DP_POD': cr['pod_dp'],
               'ConvGRU_CSI': cr['csi_gru'], 'DP_CSI': cr['csi_dp']}
              for h, cr in case_results.items()]
pd.DataFrame(case_table).to_csv(output_dir / 'tables' / f'case_study_{drought_str}.csv', index=False)
print('\nSection 16 complete.')


## Section 17: Future Forecast

In [ ]:
# RULE: best-validation-loss seed (best_models[h]) - see configs/best_seed_map.json
print('=== Section 17: Future Forecast ===')

last_t    = N_MONTHS - 1
last_date = all_dates[last_t]
print(f'Last available data: {last_date[0]}-{last_date[1]:02d} (index {last_t})')

X_future = feat[last_t - cfg['SEQ_LEN'] + 1 : last_t + 1][None, ...]
print(f'Input sequence shape: {X_future.shape}')

future_rows = []
future_pred_maps = {}

for h in cfg['HORIZONS']:
    model = best_models[h]; alpha = damped_alphas[h]

    target_year  = last_date[0]
    target_month = last_date[1] + h
    if target_month > 12:
        target_year  += target_month // 12
        target_month  = target_month % 12
        if target_month == 0:
            target_month = 12; target_year -= 1
    target_str = f'{target_year}-{target_month:02d}'

    # FIX: use spi6[last_t] (the most recent available SPI, not last_t-1)
    persistence  = spi6[last_t]
    climatology  = clim[target_month - 1]
    baseline_fut = alpha * persistence + (1 - alpha) * climatology

    pred_delta = model.predict(X_future, batch_size=1, verbose=0)
    pred_abs   = baseline_fut + pred_delta[0]
    pred_abs[~ROI_MASK] = np.nan
    future_pred_maps[h] = pred_abs.copy()

    roi_vals    = pred_abs[ROI_MASK]; roi_vals = roi_vals[np.isfinite(roi_vals)]
    drought_pct = (roi_vals < cfg['DROUGHT_THRESHOLD']).mean() * 100
    severe_pct  = (roi_vals < -1.5).mean() * 100
    extreme_pct = (roi_vals < -2.0).mean() * 100

    future_rows.append({'horizon': h, 'target_date': target_str,
                        'mean_SPI6': float(np.mean(roi_vals)), 'median_SPI6': float(np.median(roi_vals)),
                        'std_SPI6': float(np.std(roi_vals)), 'min_SPI6': float(np.min(roi_vals)),
                        'max_SPI6': float(np.max(roi_vals)),
                        'pct_drought_moderate': float(drought_pct),
                        'pct_drought_severe':   float(severe_pct),
                        'pct_drought_extreme':  float(extreme_pct)})
    print(f'H={h} -> {target_str}: Mean={np.mean(roi_vals):.3f}  Drought={drought_pct:.1f}%  Severe={severe_pct:.1f}%  Extreme={extreme_pct:.1f}%')

future_df = pd.DataFrame(future_rows)
future_df.to_csv(output_dir / 'tables' / 'future_forecast.csv', index=False)
print('\n' + future_df.to_string(index=False, float_format='%.3f'))

spi_cmap = plt.cm.RdYlBu
spi_norm = mcolors.TwoSlopeNorm(vmin=-3, vcenter=0, vmax=3)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Future SPI-6 Drought Forecast (beyond available data)', fontsize=14, fontweight='bold')
for col, h in enumerate(cfg['HORIZONS']):
    target_str = future_rows[col]['target_date']
    pred_map   = future_pred_maps[h].copy()
    im = axes[0, col].imshow(pred_map, cmap=spi_cmap, norm=spi_norm, aspect='auto')
    axes[0, col].set_title(f'Predicted SPI-6: {target_str} (H={h})', fontsize=11); axes[0, col].axis('off')
    severity = np.full_like(pred_map, np.nan)
    severity[pred_map < -2.0] = 3
    severity[(pred_map >= -2.0) & (pred_map < -1.5)] = 2
    severity[(pred_map >= -1.5) & (pred_map < -1.0)] = 1
    severity[pred_map >= -1.0] = 0
    severity[~ROI_MASK] = np.nan
    sev_cmap = mcolors.ListedColormap(['#2166ac', '#fee08b', '#fc8d59', '#d73027'])
    axes[1, col].imshow(severity, cmap=sev_cmap, vmin=0, vmax=3, aspect='auto')
    axes[1, col].set_title(f'Drought Severity: {target_str}', fontsize=11); axes[1, col].axis('off')
plt.colorbar(im, ax=axes[0, :], fraction=0.02, pad=0.04, label='SPI-6')
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#2166ac', label='No drought (>-1)'),
                   Patch(facecolor='#fee08b', label='Moderate (-1.5 to -1)'),
                   Patch(facecolor='#fc8d59', label='Severe (-2 to -1.5)'),
                   Patch(facecolor='#d73027', label='Extreme (<-2)')]
axes[1, 2].legend(handles=legend_elements, loc='center left', bbox_to_anchor=(1.05, 0.5), fontsize=9)
plt.tight_layout()
plt.savefig(output_dir / 'figures' / 'fig14_future_forecast.png', dpi=200, bbox_inches='tight')
plt.show()

# Zoomed historical + forecast
dates_all_pd = pd.to_datetime([f'{d[0]}-{d[1]:02d}' for d in all_dates], format='%Y-%m')
roi_spi_hist = np.array([np.nanmean(spi6[t][ROI_MASK]) for t in range(N_MONTHS)])
zoom_start   = pd.to_datetime('2020-01')

fig, axes = plt.subplots(3, 1, figsize=(14, 10))
fig.suptitle('Recent History + Future Forecast (zoomed)', fontsize=14, fontweight='bold')
for ax, h in zip(axes, cfg['HORIZONS']):
    target_str  = future_rows[h - 1]['target_date']
    future_spi  = float(np.nanmean(future_pred_maps[h][ROI_MASK]))
    future_date = pd.to_datetime(target_str, format='%Y-%m')
    mask_zoom   = dates_all_pd >= zoom_start
    ax.plot(dates_all_pd[mask_zoom], roi_spi_hist[mask_zoom], color='black', lw=2.0,
            marker='o', markersize=4, label='Observed SPI-6')
    ax.plot(future_date, future_spi, marker='*', markersize=22, color='#d50000',
            zorder=10, label=f'FORECAST {target_str}: {future_spi:.2f}')
    ax.axhline(cfg['DROUGHT_THRESHOLD'], color='red', lw=1, ls='--', alpha=0.5, label='Drought threshold')
    ax.fill_between(dates_all_pd[mask_zoom], cfg['DROUGHT_THRESHOLD'], roi_spi_hist[mask_zoom],
                    where=roi_spi_hist[mask_zoom] < cfg['DROUGHT_THRESHOLD'], alpha=0.15, color='red')
    ax.set_ylabel('SPI-6 (ROI mean)'); ax.set_title(f'H={h}', fontsize=11)
    ax.legend(loc='best', fontsize=9); ax.grid(True, alpha=0.3)
    ax.set_xlim(zoom_start, future_date + pd.DateOffset(months=2))
plt.tight_layout()
plt.savefig(output_dir / 'figures' / 'fig16_zoomed_forecast.png', dpi=200, bbox_inches='tight')
plt.show()

for h in cfg['HORIZONS']:
    target_str = future_rows[h - 1]['target_date']
    geotiff_path = output_dir / 'geotiffs' / f'forecast_{target_str}_H{h}.tif'
    band = future_pred_maps[h].copy(); band[~np.isfinite(band)] = -9999.0
    with rasterio.open(geotiff_path, 'w', driver='GTiff', height=HEIGHT, width=WIDTH, count=1,
                       dtype='float32', crs=profile['crs'], transform=profile['transform'], nodata=-9999.0) as dst:
        dst.write(band, 1)
    print(f'Saved: {geotiff_path}')

print('Section 17 complete.')


## Section 18: Export Package

In [ ]:
print('=== Section 18: Export Package ===')

cfg_serializable = {k: (list(v) if isinstance(v, tuple) else v) for k, v in cfg.items()}
with open(output_dir / 'config.json', 'w') as f:
    json.dump(cfg_serializable, f, indent=2)

with open(output_dir / 'seed_list.json', 'w') as f:
    json.dump(list(cfg['SEEDS']), f)

tables  = sorted((output_dir / 'tables').glob('*.csv'))
figures = sorted((output_dir / 'figures').glob('*.png'))
tiffs   = sorted((output_dir / 'geotiffs').glob('*.tif'))
models  = sorted((output_dir / 'models').glob('*.h5'))

manifest = {
    'generated_by': 'Final_SPI6_DroughtForecasting.ipynb',
    'config': 'config.json',
    'tables':   [str(f.name) for f in tables],
    'figures':  [str(f.name) for f in figures],
    'geotiffs': [str(f.name) for f in tiffs],
    'models':   [str(f.name) for f in models],
}
with open(output_dir / 'manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)

print('config.json, seed_list.json, manifest.json saved.')
print('\nrequirements (approximate):')
for r in ['tensorflow>=2.12', 'numpy>=1.23', 'pandas>=1.5',
          'matplotlib>=3.6', 'scipy>=1.9', 'scikit-learn>=1.1',
          'rasterio>=1.3', 'tqdm>=4.64']:
    print(f'  {r}')

print('\n' + '='*70 + '\nALL OUTPUT FILES\n' + '='*70)
print(f'\nTables ({len(tables)}):')
for f in tables: print(f'  {f.name}')
print(f'\nFigures ({len(figures)}):')
for f in figures: print(f'  {f.name}')
print(f'\nGeoTIFFs ({len(tiffs)}):')
for f in tiffs: print(f'  {f.name}')
print(f'\nModels ({len(models)}):')
for f in models: print(f'  {f.name}')
print('\n' + '='*70 + '\nExecution complete. Output files written to outputs_final_fin/.\n' + '='*70)


In [ ]:
# ============================================================
# SECTION: Package Export
# Consolidates notebook outputs into final_paper_package layout.
# Update PACKAGE_DIR below if your final_paper_package folder is elsewhere.
# ============================================================
import shutil

PACKAGE_DIR = Path(cfg['OUTPUT_DIR']).parent / 'final_paper_package'
(PACKAGE_DIR / 'metrics').mkdir(parents=True, exist_ok=True)
(PACKAGE_DIR / 'tables').mkdir(parents=True, exist_ok=True)

t = output_dir / 'tables'

def _copy(src, dst):
    if src.exists():
        shutil.copy(src, dst)
        print(f'  {dst.relative_to(PACKAGE_DIR)}')
    else:
        print(f'  SKIP (not found): {src.name}')

print('=== Package Export ===')

# metrics/per_seed_metrics.csv - per-seed metrics with actual seed values
_src = t / 'convgru_results.csv'
if not _src.exists():
    _src = t / 'seed_summary.csv'
_copy(_src, PACKAGE_DIR / 'metrics' / 'per_seed_metrics.csv')

# metrics/dm_test_results.csv
_copy(t / 'diebold_mariano.csv', PACKAGE_DIR / 'metrics' / 'dm_test_results.csv')

# metrics/event_metrics.csv
_copy(t / 'drought_categorical_metrics.csv', PACKAGE_DIR / 'metrics' / 'event_metrics.csv')

# metrics/aggregated_metrics.csv - wide format: seed std + bootstrap CI per horizon
_boot_f = t / 'metrics_bootstrap_ci95.csv'
if _boot_f.exists() and dl_df is not None:
    _boot = pd.read_csv(_boot_f)
    _agg_rows = []
    for _h in cfg['HORIZONS']:
        _bh  = _boot[_boot['horizon'] == _h].iloc[0]
        _dlh = dl_df[dl_df['horizon'] == _h]
        _row = {'horizon': _h,
                'rmse_mean': _bh['rmse_mean'], 'rmse_std': float(_dlh['rmse'].std()),
                'corr_mean': _bh['corr_mean'], 'corr_std': float(_dlh['corr'].std()),
                'pod_mean':  _bh['pod_mean'],  'pod_std':  float(_dlh['pod'].std()),
                'csi_mean':  _bh['csi_mean'],  'csi_std':  float(_dlh['csi'].std()),
                'mae_mean':  _bh['mae_mean'],  'mae_std':  float(_dlh['mae'].std()),
                'rmse_ci95_lo': _bh['rmse_ci95_lo'], 'rmse_ci95_hi': _bh['rmse_ci95_hi'],
                'corr_ci95_lo': _bh['corr_ci95_lo'], 'corr_ci95_hi': _bh['corr_ci95_hi'],
                'pod_ci95_lo':  _bh['pod_ci95_lo'],  'pod_ci95_hi':  _bh['pod_ci95_hi'],
                'csi_ci95_lo':  _bh['csi_ci95_lo'],  'csi_ci95_hi':  _bh['csi_ci95_hi'],
                'mae_ci95_lo':  _bh['mae_ci95_lo'],  'mae_ci95_hi':  _bh['mae_ci95_hi']}
        _agg_rows.append(_row)
    pd.DataFrame(_agg_rows).to_csv(PACKAGE_DIR / 'metrics' / 'aggregated_metrics.csv', index=False)
    print(f'  metrics/aggregated_metrics.csv')
else:
    print('  SKIP aggregated_metrics (no bootstrap CSV or dl_df)')

# tables/table_main_metrics.csv - fill NaN stds with 0.0 for deterministic baselines
if (t / 'comparison_all.csv').exists():
    _tm = pd.read_csv(t / 'comparison_all.csv')
    _det = ~_tm['model'].str.contains('ConvGRU_B', na=False)
    for _c in ['rmse_std', 'corr_std']:
        if _c in _tm.columns:
            _tm.loc[_det, _c] = _tm.loc[_det, _c].fillna(0.0)
    _tm.to_csv(PACKAGE_DIR / 'tables' / 'table_main_metrics.csv', index=False)
    print(f'  tables/table_main_metrics.csv')
else:
    print('  SKIP table_main_metrics (comparison_all.csv not found)')

# tables/table_baseline_comparison.csv - baselines + ridge (no ConvGRU)
if (t / 'baselines.csv').exists():
    _b = pd.read_csv(t / 'baselines.csv')
    _r = pd.read_csv(t / 'ridge_results.csv') if (t / 'ridge_results.csv').exists() else pd.DataFrame()
    pd.concat([_b, _r], ignore_index=True).to_csv(
        PACKAGE_DIR / 'tables' / 'table_baseline_comparison.csv', index=False)
    print('  tables/table_baseline_comparison.csv')

# tables/table_event_detection.csv
_copy(t / 'drought_categorical_metrics.csv', PACKAGE_DIR / 'tables' / 'table_event_detection.csv')

# tables/table_future_forecast.csv
_copy(t / 'future_forecast.csv', PACKAGE_DIR / 'tables' / 'table_future_forecast.csv')

# --- predictions/ folder - per-seed and ensemble test predictions as .npz ---
_pred_dir = PACKAGE_DIR / 'predictions'
_pred_dir.mkdir(exist_ok=True)
for _h in cfg['HORIZONS']:
    _tp = all_test_preds[_h]
    np.savez_compressed(
        _pred_dir / f'ensemble_test_preds_H{_h}.npz',
        pred_abs=_tp['pred_abs'].astype('float32'),
        obs=_tp['obs'].astype('float32'),
        baseline=_tp['baseline'].astype('float32'),
        anchor=np.array(_tp['anchor'], dtype='int32'),
    )
    for _sp in all_seed_preds[_h]:
        np.savez_compressed(
            _pred_dir / f'per_seed_test_preds_H{_h}_s{_sp["seed"]}.npz',
            pred_abs=_sp['pred_abs'].astype('float32'),
            pred_delta=_sp['pred_delta'].astype('float32'),
            obs=_sp['obs'].astype('float32'),
            baseline=_sp['baseline'].astype('float32'),
            anchor=np.array(_sp['anchor'], dtype='int32'),
        )
print(f'  predictions/  ({len(list(_pred_dir.glob("*.npz")))} npz files)')

print('Package export complete.')
